In [1]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 2-OFFICIAL — DS-Attn-UNet + ASPP on the OFFICIAL TCIA SPLIT
#   Architecture / loss / augmentation / optimiser: IDENTICAL to your CV
#   segmentation cell. Only the fold roles change (role_f -> role_of).
#   Trains only on official-training regions; the 378 official test
#   regions are never seen during training or validation.
#   Output -> predmasks_mass_official/     (predmasks_mass/ untouched)
#   Resumable: a finished fold reloads its checkpoint instead of retraining.
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
DS_WEIGHTS=[1.0,0.5,0.3,0.2]
torch.backends.cudnn.benchmark=True

LES="mass"
CSV=os.path.join(D,"unified_folds_%s.csv"%LES)
OUT=os.path.join(D,"predmasks_%s_official"%LES); os.makedirs(OUT,exist_ok=True)
CKD=os.path.join(D,"ckpt_official"); os.makedirs(CKD,exist_ok=True)
ck=lambda k: os.path.join(CKD,"seg_dsaspp_%s_official_fold%d.pth"%(LES,k))

# ═══════════════ VERIFICATION — runs before any GPU work ═══════════════
d=pd.read_csv(CSV).reset_index(drop=True)
print("="*74); print("VERIFICATION — OFFICIAL TCIA SPLIT"); print("="*74)
print("  csv                : %s"%CSV)
assert "official_split" in d.columns, "official_split column missing"
assert "msk" in d.columns and "img" in d.columns, "img/msk columns missing"
_te=d["official_split"].astype(str).str.lower().str.contains("test")
print("  regions            : %d  (official train %d | official test %d)"
      %(len(d),(~_te).sum(),_te.sum()))
print("  patients           : train %d | test %d"
      %(d.patient_id[~_te].nunique(), d.patient_id[_te].nunique()))
ov=len(set(d.patient_id[~_te]) & set(d.patient_id[_te]))
print("  patients in BOTH   : %d"%ov); assert ov==0, "patient overlap in the official split"
assert (~_te).sum()==1318 and _te.sum()==378, \
       "expected 1318/378 official regions, got %d/%d"%((~_te).sum(),_te.sum())
for k in range(5):
    c="role_of%d"%k
    assert c in d.columns, "%s missing - run CELL A first"%c
    r=d[c]
    assert ((r=="test")==_te).all(), "%s test set != official test"%c
    assert not (r.isin(["train","val"]) & _te).any(), "OFFICIAL TEST REGION IN TRAINING (%s)"%c
    assert len(set(d.patient_id[r=="train"]) & set(d.patient_id[r=="test"]))==0, "%s train/test patient leak"%c
    assert len(set(d.patient_id[r=="val"])   & set(d.patient_id[r=="test"]))==0, "%s val/test patient leak"%c
    assert len(set(d.patient_id[r=="train"]) & set(d.patient_id[r=="val"]))==0,  "%s train/val patient leak"%c
    print("  %-9s : train %4d | val %4d | test %3d   (test == official test)"
          %(c,(r=="train").sum(),(r=="val").sum(),(r=="test").sum()))
nm=d["msk"].astype(str).apply(os.path.exists).sum(); ni=d["img"].astype(str).apply(os.path.exists).sum()
print("  files on disk      : img %d/%d | msk %d/%d"%(ni,len(d),nm,len(d)))
assert nm==len(d) and ni==len(d), "some img/msk files are missing"
print("  VERIFIED: training uses official-train regions only.")
print("="*74)

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))
class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2)
        s.bn=ASPP(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out =nn.Conv2d(b,2,1)
        s.ds2=nn.Conv2d(b*2,2,1); s.ds3=nn.Conv2d(b*4,2,1); s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        main=s.out(d1)
        if s.training: return main, s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return main

def tversky_ce(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())
def ds_loss(outs,t):
    main,o2,o3,o4=outs
    L=DS_WEIGHTS[0]*tversky_ce(main,t)
    for w,o in zip(DS_WEIGHTS[1:],[o2,o3,o4]):
        td=F.interpolate(t.unsqueeze(1).float(),size=o.shape[2:],mode="nearest").squeeze(1).long()
        L=L+w*tversky_ce(o,td)
    return L
@torch.no_grad()
def sc(net,dd):
    net.eval(); ld=DataLoader(DS(dd,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),
                prec=R.prec.mean(),rec=R.rec.mean())
@torch.no_grad()
def probs(net,dd):
    net.eval(); ld=DataLoader(DS(dd,False),batch_size=BATCH,shuffle=False,num_workers=0)
    out=np.zeros((len(dd),IMG,IMG),np.float32); pos=0
    for x,_,_ in ld:
        x=x.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        p=torch.softmax(o.float(),1)[:,1].cpu().numpy()
        out[pos:pos+len(p)]=p; pos+=len(p)
    return out
def save_mask(row, prob):
    m=(prob>THR).astype(np.uint8)
    cv2.imwrite(os.path.join(OUT, os.path.basename(row["img"]).replace("_img.png","")+"_pred.png"),
                cv2.resize(m*255,(512,512),interpolation=cv2.INTER_NEAREST))
    gt=cv2.imread(row["msk"],cv2.IMREAD_GRAYSCALE)
    gt=(cv2.resize(gt,(IMG,IMG),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
    tp=(m*gt).sum(); fp=(m*(1-gt)).sum(); fn=((1-m)*gt).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

test_i=np.where(_te.values)[0]
test_sum=np.zeros((len(test_i),IMG,IMG),np.float32)
oof=np.full(len(d),np.nan); fold_dice=[]

for k in range(5):
    role=d["role_of%d"%k]
    tr=d[role=="train"]; va=d[role=="val"]; te=d[role=="test"]
    net=DSAttnUNet().to(DEV)
    if os.path.exists(ck(k)):
        net.load_state_dict({q:v.to(DEV) for q,v in torch.load(ck(k),map_location="cpu").items()})
        print("\n### fold %d — checkpoint found, skipping training"%k)
    else:
        print("\n### fold %d | train %d x%d | val %d | test %d"%(k,len(tr),MULT,len(va),len(te)))
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
        scaler=torch.amp.GradScaler(); opt=torch.optim.Adam(net.parameters(),lr=LR)
        sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
        best,bs,ni=0.,None,0
        for ep in range(1,EPOCHS+1):
            net.train(); tot=0.; nb=0
            for x,y,_ in tl:
                x=x.to(DEV); y=y.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"): outs=net(x); l=ds_loss(outs,y)
                scaler.scale(l).backward(); scaler.step(opt); scaler.update()
                tot+=l.item(); nb+=1
            vd=sc(net,va)["dice"]; sch.step(vd)
            if vd>best: best=vd; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            print("  ep %2d | loss %.4f | val-Dice %.4f%s"%(ep,tot/max(nb,1),vd," *" if vd==best else ""))
            if ni>=10: print("  early stop"); break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        torch.save({q:v.cpu() for q,v in net.state_dict().items()}, ck(k))
        print("  fold %d trained in %.0fs — checkpoint saved"%(k,time.time()-t0))

    m=sc(net,te); fold_dice.append(m["dice"])
    print("  official TEST Dice %.4f (median %.4f) | IoU %.4f | P %.3f | R %.3f"
          %(m["dice"],m["median"],m["iou"],m["prec"],m["rec"]))
    pv=probs(net,va)
    for i,gi in enumerate(va.index.values):
        oof[gi]=save_mask(d.iloc[gi], pv[i])
    test_sum += probs(net,te)
    del net; torch.cuda.empty_cache()

print("\nwriting 5-fold averaged masks for the %d official TEST regions..."%len(test_i))
for i,gi in enumerate(test_i):
    oof[gi]=save_mask(d.iloc[gi], test_sum[i]/5.0)

assert not np.isnan(oof).any(), "%d regions got no mask"%int(np.isnan(oof).sum())
d["oof_dice_official"]=oof
d.to_csv(CSV,index=False)

tr_m=oof[~_te.values].mean(); te_m=oof[_te.values].mean()
print("\n"+"="*74)
print("SEGMENTATION ON THE OFFICIAL TCIA SPLIT — DS-Attn-UNet + ASPP")
print("="*74)
print("  per-fold official TEST Dice : %s"%[round(x,4) for x in fold_dice])
print("  MEAN over folds             : %.4f +/- %.4f"%(np.mean(fold_dice),np.std(fold_dice)))
print("  5-fold averaged masks, TEST : %.4f   (n=%d)  <- report this"%(te_m,_te.sum()))
print("  out-of-fold masks,   TRAIN  : %.4f   (n=%d)"%(tr_m,(~_te).sum()))
print("  CV reference                : 0.900")
print("  masks -> %s"%OUT)
print("="*74)

VERIFICATION — OFFICIAL TCIA SPLIT
  csv                : /root/autodl-tmp/CBIS/unified_folds_mass.csv
  regions            : 1696  (official train 1318 | official test 378)
  patients           : train 691 | test 201
  patients in BOTH   : 0
  role_of0  : train 1054 | val  264 | test 378   (test == official test)
  role_of1  : train 1054 | val  264 | test 378   (test == official test)
  role_of2  : train 1053 | val  265 | test 378   (test == official test)
  role_of3  : train 1057 | val  261 | test 378   (test == official test)
  role_of4  : train 1054 | val  264 | test 378   (test == official test)
  files on disk      : img 1696/1696 | msk 1696/1696
  VERIFIED: training uses official-train regions only.

### fold 0 | train 1054 x8 | val 264 | test 378
  ep  1 | loss 0.3624 | val-Dice 0.8602 *
  ep  2 | loss 0.2944 | val-Dice 0.8461
  ep  3 | loss 0.2759 | val-Dice 0.8724 *
  ep  4 | loss 0.2654 | val-Dice 0.8743 *
  ep  5 | loss 0.2550 | val-Dice 0.8735
  ep  6 | loss 0.2435 | val-D

In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 11 — DOES CLASSIFIER PERFORMANCE DEPEND ON MASK QUALITY?
#   Stratifies the 378 official test regions by their ACTUAL segmentation
#   Dice and reports AUC / accuracy inside each stratum.
#   Uses real segmentation failures, not synthetic degradation.
#   No GPU, no inference. ~30 s.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")

# ── PART 1 : find the per-region Dice column ──────────────────────────────
print("="*82); print("PART 1 — PER-REGION MASK QUALITY"); print("="*82)
dc=[c for c in d.columns if "dice" in c.lower()]
print("  dice-like columns in unified_folds_mass.csv: %s" % dc)
assert dc, "no per-region Dice column found — was PHASE 2 run and the csv written back?"
DC=dc[0] if "oof_dice_official" not in dc else "oof_dice_official"
d["dice"]=pd.to_numeric(d[DC],errors="coerce")
print("  using column: %s" % DC)
print("  coverage: %d/%d rows have a Dice value" % (d.dice.notna().sum(),len(d)))

def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
PTR=load("cv_mass_twostream_officialtrain_oof.csv")
PTE=load("cv_mass_twostream_officialsplit_oof.csv")
B=d.set_index("_k")

te=d[d.sp.eq("test") & d._k.isin(PTE.index) & d.dice.notna()].copy()
te["p"]=PTE.loc[te._k].values
tr=d[d.sp.eq("train") & d._k.isin(PTR.index)].copy(); tr["p"]=PTR.loc[tr._k].values
print("  test regions with both a mask Dice and a prediction: %d" % len(te))
q=te.dice.describe(percentiles=[.05,.10,.25,.5,.75,.90])
print("  test mask Dice: min %.3f | 5%% %.3f | 25%% %.3f | median %.3f | 75%% %.3f | max %.3f | mean %.4f"
      % (q["min"],q["5%"],q["25%"],q["50%"],q["75%"],q["max"],te.dice.mean()))

# ── PART 2 : is segmentation worse on malignant lesions? ──────────────────
print("\n"+"="*82); print("PART 2 — IS MASK QUALITY RELATED TO THE LABEL?"); print("="*82)
print("  mean Dice  malignant %.4f (n=%d)   benign %.4f (n=%d)   difference %+.4f"
      % (te.dice[te.y==1].mean(),int((te.y==1).sum()),
         te.dice[te.y==0].mean(),int((te.y==0).sum()),
         te.dice[te.y==1].mean()-te.dice[te.y==0].mean()))
print("  (a large negative difference would mean segmentation fails more on cancers,")
print("   which would make low-Dice cases systematically harder, not just noisier)")

# ── threshold machinery ───────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

# thresholds frozen from TRAIN, exactly as in the main result
TAU,G0=fit_bir(tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int),FLOOR)
print("\n  frozen thresholds from TRAIN: %s" % {int(k):round(v,3) for k,v in sorted(TAU.items())})

def block(t,name,dcol="dice"):
    y=t.y.values.astype(int); p=t.p.values; a=t.a.values.astype(int)
    if len(np.unique(y))<2: return None
    yh=ap(p,a,TAU,G0)
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(n=len(y),mal=int(y.sum()),dice=t[dcol].mean(),
                auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),
                sens=tp/max(tp+fn,1),spec=tn/max(tn+fp,1))

def report(t,bins,labels,name,dcol="dice"):
    print("\n"+"="*82); print("%s — stratified by ACTUAL mask Dice" % name); print("="*82)
    print("  %-16s %-5s %-6s %-8s %-8s %-8s %-6s %s"
          % ("stratum","n","malig","meanDice","AUC","acc","sens","spec"))
    t=t.copy(); t["bin"]=pd.cut(t[dcol],bins=bins,labels=labels,include_lowest=True)
    for lb in labels:
        s=t[t.bin==lb]
        if len(s)<8: print("  %-16s %-5d  (too few to evaluate)" % (lb,len(s))); continue
        r=block(s,lb,dcol)
        if r is None: print("  %-16s %-5d  (single class)" % (lb,len(s))); continue
        print("  %-16s %-5d %-6d %-8.4f %-8.4f %-8.4f %-6.3f %.3f"
              % (lb,r["n"],r["mal"],r["dice"],r["auc"],r["acc"],r["sens"],r["spec"]))
    r=block(t,"ALL",dcol)
    print("  %-16s %-5d %-6d %-8.4f %-8.4f %-8.4f %-6.3f %.3f"
          % ("ALL",r["n"],r["mal"],r["dice"],r["auc"],r["acc"],r["sens"],r["spec"]))

# ── PART 3 : ROI level ────────────────────────────────────────────────────
report(te,[0,.70,.80,.90,1.01],["Dice <0.70","0.70-0.80","0.80-0.90","0.90+"],
       "PART 3 — ROI LEVEL (n=%d)" % len(te))
qs=te.dice.quantile([0,.25,.5,.75,1.0]).values
report(te,list(np.unique(qs)),["Q1 worst","Q2","Q3","Q4 best"][:len(np.unique(qs))-1],
       "PART 3b — ROI LEVEL, Dice quartiles")

# ── PART 4 : lesion level (mean Dice of the lesion's regions) ─────────────
les=(te.groupby("lesion_key")
       .agg(p=("p","mean"),y=("y","max"),a=("a","max"),dice=("dice","mean")).reset_index())
report(les,[0,.70,.80,.90,1.01],["Dice <0.70","0.70-0.80","0.80-0.90","0.90+"],
       "PART 4 — LESION LEVEL (n=%d)" % len(les))

# ── PART 5 : does poor segmentation predict classifier error? ─────────────
print("\n"+"="*82); print("PART 5 — CORRELATION BETWEEN MASK QUALITY AND ERROR"); print("="*82)
y=te.y.values.astype(int); err=np.abs(te.p.values-y)
c=np.corrcoef(te.dice.values,err)[0,1]
yh=ap(te.p.values,te.a.values.astype(int),TAU,G0); wrong=(yh!=y)
print("  correlation( mask Dice , |prob - label| )        %+.4f" % c)
print("  mean Dice on CORRECT predictions   %.4f  (n=%d)" % (te.dice[~wrong].mean(),int((~wrong).sum())))
print("  mean Dice on WRONG   predictions   %.4f  (n=%d)" % (te.dice[wrong].mean(),int(wrong.sum())))
print("  difference %+.4f" % (te.dice[~wrong].mean()-te.dice[wrong].mean()))
print("\n  a correlation near zero and a small difference mean classifier errors are NOT")
print("  driven by segmentation failure — which is the answer the reviewer is asking for.")

PART 1 — PER-REGION MASK QUALITY
  dice-like columns in unified_folds_mass.csv: ['oof_dice', 'oof_dice_tta', 'oof_dice_official']
  using column: oof_dice_official
  coverage: 1696/1696 rows have a Dice value
  test regions with both a mask Dice and a prediction: 378
  test mask Dice: min 0.413 | 5% 0.784 | 25% 0.888 | median 0.925 | 75% 0.942 | max 0.978 | mean 0.9065

PART 2 — IS MASK QUALITY RELATED TO THE LABEL?
  mean Dice  malignant 0.9050 (n=147)   benign 0.9075 (n=231)   difference -0.0025
  (a large negative difference would mean segmentation fails more on cancers,
   which would make low-Dice cases systematically harder, not just noisier)

  frozen thresholds from TRAIN: {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}

PART 3 — ROI LEVEL (n=378) — stratified by ACTUAL mask Dice
  stratum          n     malig  meanDice AUC      acc      sens   spec
  Dice <0.70       7      (too few to evaluate)
  0.70-0.80        15    7      0.7614   0.8393   0.8000   0.714  0.875


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 12 — CONFIDENCE INTERVALS ON THE STRATIFIED ANALYSIS
#   Bootstrap CI per Dice quartile, exact binomial CI on the sub-0.80 tail,
#   and a CI on the Dice-vs-error correlation.  ~20 s.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import beta, pearsonr
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
RNG=np.random.default_rng(7); NB=4000
TAU={0:.515,1:.44,2:.44,3:.495,4:.455,5:.01}; G0=.44

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
d["dice"]=pd.to_numeric(d["oof_dice_official"],errors="coerce")
m=pd.read_csv(os.path.join(D,"cv_mass_twostream_officialsplit_oof.csv")); m["_k"]=m["img"].map(stem)
t=d[d.sp.eq("test")].merge(m[["_k","prob"]].drop_duplicates("_k"),on="_k").dropna(subset=["dice"])
print("test regions: %d" % len(t))

def boot_auc(y,p,nb=NB):
    o=[]
    for _ in range(nb):
        i=RNG.integers(0,len(y),len(y))
        if len(np.unique(y[i]))<2: continue
        o.append(roc_auc_score(y[i],p[i]))
    return np.percentile(o,2.5),np.percentile(o,97.5)
def wilson(k,n):
    if n==0: return (np.nan,np.nan)
    lo=beta.ppf(.025,k,n-k+1) if k>0 else 0.0
    hi=beta.ppf(.975,k+1,n-k) if k<n else 1.0
    return lo,hi
pred=lambda p,a:(p>=np.array([TAU.get(int(g),G0) for g in a])).astype(int)

print("\n"+"="*84); print("PER-QUARTILE CONFIDENCE INTERVALS"); print("="*84)
t=t.copy(); t["q"]=pd.qcut(t.dice,4,labels=["Q1 worst","Q2","Q3","Q4 best"])
print("  %-10s %-5s %-13s %-22s %s" % ("stratum","n","Dice range","AUC [95% CI]","accuracy [95% CI]"))
for lb in ["Q1 worst","Q2","Q3","Q4 best"]:
    s=t[t.q==lb]; y=s.y.values.astype(int); p=s.prob.values
    lo,hi=boot_auc(y,p); k=int((pred(p,s.a.values)==y).sum()); n=len(s)
    al,ah=wilson(k,n)
    print("  %-10s %-5d %.3f-%.3f  %.4f [%.3f-%.3f]    %.3f [%.3f-%.3f]"
          % (lb,n,s.dice.min(),s.dice.max(),roc_auc_score(y,p),lo,hi,k/n,al,ah))
y=t.y.values.astype(int); p=t.prob.values
lo,hi=boot_auc(y,p); k=int((pred(p,t.a.values)==y).sum())
al,ah=wilson(k,len(t))
print("  %-10s %-5d %.3f-%.3f  %.4f [%.3f-%.3f]    %.3f [%.3f-%.3f]"
      % ("ALL",len(t),t.dice.min(),t.dice.max(),roc_auc_score(y,p),lo,hi,k/len(t),al,ah))

print("\n"+"="*84); print("THE SUB-0.80 TAIL, POOLED"); print("="*84)
for lab,sel in (("Dice < 0.80",t.dice<0.80),("Dice < 0.70",t.dice<0.70),
                ("Dice >= 0.80",t.dice>=0.80)):
    s=t[sel]; n=len(s)
    if n==0: continue
    yy=s.y.values.astype(int); k=int((pred(s.prob.values,s.a.values)==yy).sum())
    al,ah=wilson(k,n)
    au=roc_auc_score(yy,s.prob.values) if len(np.unique(yy))>1 else float("nan")
    print("  %-14s n=%-4d malig %-3d  meanDice %.4f  acc %.3f [%.3f-%.3f]  AUC %.4f"
          % (lab,n,int(yy.sum()),s.dice.mean(),k/n,al,ah,au))
print("  (a wide interval here IS the answer: it states the limit of what this cohort can show)")

print("\n"+"="*84); print("CORRELATION, FULL SAMPLE"); print("="*84)
err=np.abs(p-y); r,pv=pearsonr(t.dice.values,err)
z=np.arctanh(r); se=1/np.sqrt(len(t)-3)
print("  corr(Dice, |prob-label|)  r = %+.4f   95%% CI [%+.4f, %+.4f]   p = %.3f"
      % (r,np.tanh(z-1.96*se),np.tanh(z+1.96*se),pv))
print("  n = %d — this is the properly powered statistic, unlike the quartile split" % len(t))

test regions: 378

PER-QUARTILE CONFIDENCE INTERVALS
  stratum    n     Dice range    AUC [95% CI]           accuracy [95% CI]
  Q1 worst   95    0.413-0.888  0.8672 [0.788-0.935]    0.811 [0.717-0.884]
  Q2         94    0.888-0.925  0.8903 [0.817-0.948]    0.809 [0.714-0.882]
  Q3         94    0.926-0.942  0.8717 [0.785-0.947]    0.819 [0.726-0.891]
  Q4 best    95    0.942-0.978  0.8855 [0.805-0.951]    0.800 [0.705-0.875]
  ALL        378   0.413-0.978  0.8769 [0.839-0.911]    0.810 [0.766-0.848]

THE SUB-0.80 TAIL, POOLED
  Dice < 0.80    n=22   malig 10   meanDice 0.7095  acc 0.864 [0.651-0.971]  AUC 0.8917
  Dice < 0.70    n=7    malig 3    meanDice 0.5983  acc 1.000 [0.590-1.000]  AUC 1.0000
  Dice >= 0.80   n=356  malig 137  meanDice 0.9187  acc 0.806 [0.761-0.846]  AUC 0.8818
  (a wide interval here IS the answer: it states the limit of what this cohort can show)

CORRELATION, FULL SAMPLE
  corr(Dice, |prob-label|)  r = +0.0062   95% CI [-0.0947, +0.1070]   p = 0.904
  n = 3

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 13 — WHERE DOES THE DECISION-LAYER GAIN COME FROM?
#   A  per-category decomposition of the decision changes
#   B  evaluation on non-BI-RADS-5 cases (thresholds fitted on everything)
#   C  BI-RADS 5 removed from train AND test, both systems refitted
#   D  paired bootstrap CI on the accuracy GAIN
#   E  how many malignant BI-RADS 5 cases the 0.01 threshold rescued
#   No GPU. ~2 min.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15; RNG=np.random.default_rng(7); NB=3000

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
PTR,PTE=load("cv_mass_twostream_officialtrain_oof.csv"),load("cv_mass_twostream_officialsplit_oof.csv")
TR=d[d.sp.eq("train")&d._k.isin(PTR.index)].copy(); TR["p"]=PTR.loc[TR._k].values
TE=d[d.sp.eq("test") &d._k.isin(PTE.index)].copy(); TE["p"]=PTE.loc[TE._k].values

def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TPc=int(((yh==1)&(y==1)).sum()); TNc=int(((yh==0)&(y==0)).sum())
    if TPc/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TPc-int((cur&(y[i]==1)).sum()); bTN=TNc-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TPc+TNc)/N+1e-12:
                tau[g]=float(GRID[j]); TPc,TNc=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TPc+TNc)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

def agg(t,key):
    if key is None: return t.reset_index(drop=True)
    g=t.groupby(key,sort=True)
    o=g.agg(p=("p","mean"),y=("y","max"),a=("a","max")).reset_index()
    o["patient_id"]=g["patient_id"].first().values; return o

for LVL,KEY in (("ROI",None),("LESION","lesion_key")):
    tr,te=agg(TR,KEY),agg(TE,KEY)
    ytr,ptr,atr=tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int)
    yte,pte,ate=te.y.values.astype(int),te.p.values,te.a.values.astype(int)
    pat=te.patient_id.values
    g1=fit_global(ytr,ptr,FLOOR); tau,g0=fit_bir(ytr,ptr,atr,FLOOR)
    yh1=(pte>=g1).astype(int); yh2=ap(pte,ate,tau,g0)
    print("\n"+"#"*84); print("# %s LEVEL — n=%d | global threshold %.3f | per-category %s"
          % (LVL,len(te),g1,{int(k):round(v,3) for k,v in sorted(tau.items())})); print("#"*84)

    # ── A : per-category decomposition ────────────────────────────────────
    print("\nA. WHERE THE DECISION CHANGES HAPPEN")
    print("  %-8s %-5s %-6s %-8s %-9s %-9s %-9s %s"
          % ("BI-RADS","n","malig","thresh","flipped","newly OK","newly bad","net correct"))
    tot=0
    for g in sorted(set(ate)):
        m=ate==g; n=int(m.sum())
        fl_=int((yh1[m]!=yh2[m]).sum())
        ok1=(yh1[m]==yte[m]); ok2=(yh2[m]==yte[m])
        gain=int((~ok1&ok2).sum()); loss=int((ok1&~ok2).sum()); tot+=gain-loss
        print("  %-8d %-5d %-6d %-8.3f %-9d %-9d %-9d %+d"
              % (g,n,int(yte[m].sum()),tau.get(int(g),g0),fl_,gain,loss,gain-loss))
    print("  %-8s %-5d %-6d %-8s %-9d %-9s %-9s %+d  -> %+.2f accuracy points"
          % ("TOTAL",len(te),int(yte.sum()),"",int((yh1!=yh2).sum()),"","",tot,100*tot/len(te)))

    # ── E : the BI-RADS 5 rescue count ────────────────────────────────────
    m5=ate==5
    resc=int(((yh1==0)&(yh2==1)&(yte==1)&m5).sum())
    hurt=int(((yh1==0)&(yh2==1)&(yte==0)&m5).sum())
    print("\nE. BI-RADS 5 SPECIFICALLY")
    print("   n=%d, malignant %d (%.0f%%)" % (int(m5.sum()),int(yte[m5].sum()),100*yte[m5].mean()))
    print("   malignant cases the 0.01 threshold RESCUED from the global cut : %d" % resc)
    print("   benign cases it newly called malignant                         : %d" % hurt)
    print("   net correct from BI-RADS 5 alone                               : %+d" % (resc-hurt))

    # ── B : evaluate on non-BI-RADS-5 only, thresholds fitted on everything
    keep=~m5
    a1=(yh1[keep]==yte[keep]).mean(); a2=(yh2[keep]==yte[keep]).mean()
    print("\nB. EVALUATED ON NON-BI-RADS-5 ONLY (thresholds fitted on all data)")
    print("   n=%d   global %.4f   per-category %.4f   gain %+.2f points"
          % (int(keep.sum()),a1,a2,100*(a2-a1)))

    # ── C : BI-RADS 5 removed from train AND test, both refitted ──────────
    ktr=atr!=5
    if ktr.sum()>50 and len(np.unique(ytr[ktr]))>1:
        g1c=fit_global(ytr[ktr],ptr[ktr],FLOOR)
        tauc,g0c=fit_bir(ytr[ktr],ptr[ktr],atr[ktr],FLOOR)
        h1=(pte[keep]>=g1c).astype(int); h2=ap(pte[keep],ate[keep],tauc,g0c)
        c1=(h1==yte[keep]).mean(); c2=(h2==yte[keep]).mean()
        print("\nC. BI-RADS 5 REMOVED FROM TRAIN AND TEST, BOTH SYSTEMS REFITTED")
        print("   thresholds %s" % {int(k):round(v,3) for k,v in sorted(tauc.items())})
        print("   n=%d   global %.4f   per-category %.4f   gain %+.2f points"
              % (int(keep.sum()),c1,c2,100*(c2-c1)))

    # ── D : paired bootstrap CI on the GAIN ───────────────────────────────
    print("\nD. PAIRED BOOTSTRAP CI ON THE ACCURACY GAIN (clustered by patient)")
    ps=np.unique(pat)
    for lab,sel in (("all cases",np.ones(len(te),bool)),("excluding BI-RADS 5",keep)):
        o=[]
        for _ in range(NB):
            ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
            ix=ix[sel[ix]]
            if len(ix)<20: continue
            o.append((yh2[ix]==yte[ix]).mean()-(yh1[ix]==yte[ix]).mean())
        o=np.array(o)
        print("   %-22s gain %+.2f pts  95%% CI [%+.2f, %+.2f]  P(gain>0) = %.3f"
              % (lab,100*o.mean(),100*np.percentile(o,2.5),100*np.percentile(o,97.5),(o>0).mean()))


####################################################################################
# ROI LEVEL — n=378 | global threshold 0.440 | per-category {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}
####################################################################################

A. WHERE THE DECISION CHANGES HAPPEN
  BI-RADS  n     malig  thresh   flipped   newly OK  newly bad net correct
  0        33    3      0.515    5         5         0         +5
  1        2     2      0.440    0         0         0         +0
  2        14    1      0.440    0         0         0         +0
  3        85    4      0.495    13        12        1         +11
  4        169   67     0.455    7         5         2         +3
  5        75    70     0.010    3         2         1         +1
  TOTAL    378   147             28                            +20  -> +5.29 accuracy points

E. BI-RADS 5 SPECIFICALLY
   n=75, malignant 70 (93%)
   malignant cases the 0.01 threshold RESCUED from th

In [4]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 14 — PAIRED CI: your layer (S2) versus the fusion baseline (S3)
#   Same 0.90 sensitivity constraint. Clustered bootstrap by patient.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15; RNG=np.random.default_rng(7); NB=3000
LOGIT=lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
for c in ("density","subtlety","side"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d.density,errors="coerce");  d["ds"]=d.ds.fillna(d.ds.median())
d["sb"]=pd.to_numeric(d.subtlety,errors="coerce"); d["sb"]=d.sb.fillna(d.sb.median())
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
TR=d[d.sp.eq("train")].copy(); TR["p"]=load("cv_mass_twostream_officialtrain_oof.csv").loc[TR._k].values
TE=d[d.sp.eq("test")].copy();  TE["p"]=load("cv_mass_twostream_officialsplit_oof.csv").loc[TE._k].values

def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    A=int(((yh==1)&(y==1)).sum()); B=int(((yh==0)&(y==0)).sum())
    if A/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bA=A-int((cur&(y[i]==1)).sum()); bB=B-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bA+tg+bB+ng)/N; se=(bA+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(A+B)/N+1e-12:
                tau[g]=float(GRID[j]); A,B=int(bA+tg[j]),int(bB+ng[j]); mv=True
        if not mv: break
    return tau,(A+B)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
CATS=[0,1,2,3,4,5]
def fuse(tr,te):
    X=lambda t: np.column_stack([LOGIT(t.p.values)]+
        [(t.a.values==g).astype(float) for g in CATS]+[t.ds.values,t.sb.values])
    Xtr,Xte=X(tr),X(te); mu,sd=Xtr.mean(0),Xtr.std(0)+1e-9
    lr=LogisticRegression(C=0.05,max_iter=5000).fit((Xtr-mu)/sd,tr.y.values)
    return lr.predict_proba((Xtr-mu)/sd)[:,1],lr.predict_proba((Xte-mu)/sd)[:,1]
def roll(t,key):
    if key is None: return t.reset_index(drop=True)
    sp=dict(p=("p","mean"),y=("y","max"),a=("a","max"),ds=("ds","first"),sb=("sb","first"))
    if key!="patient_id": sp["patient_id"]=("patient_id","first")
    o=t.groupby(key,sort=True).agg(**sp).reset_index()
    if "patient_id" not in o.columns: o["patient_id"]=o[key].values
    return o

print("="*88); print("S2 (image-only + per-BI-RADS) vs S3 (fusion + one threshold), floor %.2f" % FLOOR)
print("="*88)
print("  %-8s %-5s %-9s %-9s %-9s %-24s %s" % ("unit","n","S2 acc","S3 acc","diff","95% CI on diff","P(S2>S3)"))
for nm,key in (("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")):
    tr,te=roll(TR,key),roll(TE,key)
    ytr,ptr,atr=tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int)
    yte,pte,ate=te.y.values.astype(int),te.p.values,te.a.values.astype(int)
    pat=te.patient_id.values
    ztr,zte=fuse(tr,te)
    tau,g0=fit_bir(ytr,ptr,atr,FLOOR); g3=fit_global(ytr,ztr,FLOOR)
    h2=ap(pte,ate,tau,g0); h3=(zte>=g3).astype(int)
    a2,a3=(h2==yte).mean(),(h3==yte).mean()
    ps=np.unique(pat); o=[]
    for _ in range(NB):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        o.append((h2[ix]==yte[ix]).mean()-(h3[ix]==yte[ix]).mean())
    o=np.array(o)
    print("  %-8s %-5d %-9.4f %-9.4f %+-9.4f [%+.4f, %+.4f]        %.3f"
          % (nm,len(te),a2,a3,a2-a3,np.percentile(o,2.5),np.percentile(o,97.5),(o>0).mean()))

S2 (image-only + per-BI-RADS) vs S3 (fusion + one threshold), floor 0.90
  unit     n     S2 acc    S3 acc    diff      95% CI on diff           P(S2>S3)
  ROI      378   0.8095    0.7910    +0.0185   [-0.0099, +0.0477]        0.881
  LESION   223   0.8565    0.8386    +0.0179   [-0.0046, +0.0444]        0.901
  BREAST   210   0.8476    0.8381    +0.0095   [-0.0141, +0.0333]        0.722
  PATIENT  201   0.8507    0.8358    +0.0149   [-0.0100, +0.0448]        0.832


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 15 — IS THE OFFICIAL TEST SET EASIER THAN THE TRAINING PARTITION?
#   PART 1  case-mix comparison, official train vs official test
#   PART 2  statistical tests on each difference
#   PART 3  the clean test: same CV models, AUC on official-test patients
#           vs official-train patients
#   No GPU. ~20 s.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
for c in ("density","subtlety"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d.density,errors="coerce")
d["sb"]=pd.to_numeric(d.subtlety,errors="coerce")
d["dice"]=pd.to_numeric(d.get("oof_dice_official"),errors="coerce")
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
TR,TE=d[d.sp.eq("train")],d[d.sp.eq("test")]

print("="*84); print("PART 1 — CASE MIX: OFFICIAL TRAIN vs OFFICIAL TEST"); print("="*84)
print("  %-26s %-14s %-14s %s" % ("variable","train","test","difference"))
print("  %-26s %-14s %-14s" % ("n regions",len(TR),len(TE)))
print("  %-26s %-14.1f %-14.1f %+.1f pts" % ("malignant %",100*TR.y.mean(),100*TE.y.mean(),
                                             100*(TE.y.mean()-TR.y.mean())))
for lab,col in (("mean subtlety (1-5)","sb"),("mean density (1-4)","ds"),
                ("mean mask Dice","dice")):
    a,b=TR[col].dropna(),TE[col].dropna()
    if len(a)==0 or len(b)==0: continue
    print("  %-26s %-14.3f %-14.3f %+.3f" % (lab,a.mean(),b.mean(),b.mean()-a.mean()))

print("\n  BI-RADS distribution (%% of partition)")
print("  %-10s %-10s %-10s %s" % ("BI-RADS","train","test","difference"))
for g in sorted(set(d.a)):
    pa,pb=100*(TR.a==g).mean(),100*(TE.a==g).mean()
    print("  %-10d %-10.1f %-10.1f %+.1f" % (g,pa,pb,pb-pa))

print("\n"+"="*84); print("PART 2 — ARE THE DIFFERENCES SIGNIFICANT?"); print("="*84)
for lab,col in (("subtlety","sb"),("density","ds"),("mask Dice","dice")):
    a,b=TR[col].dropna(),TE[col].dropna()
    if len(a)<10 or len(b)<10: continue
    u,p=mannwhitneyu(a,b,alternative="two-sided")
    print("  %-14s Mann-Whitney p = %.4f   %s" % (lab,p,"DIFFERENT" if p<0.05 else "no difference"))
ct=pd.crosstab(d.sp,d.a)
chi,p,_,_=chi2_contingency(ct)
print("  %-14s chi-square   p = %.4f   %s" % ("BI-RADS",p,"DIFFERENT" if p<0.05 else "no difference"))
ct2=pd.crosstab(d.sp,d.y); chi2_,p2,_,_=chi2_contingency(ct2)
print("  %-14s chi-square   p = %.4f   %s" % ("malignancy",p2,"DIFFERENT" if p2<0.05 else "no difference"))

print("\n"+"="*84); print("PART 3 — THE CLEAN TEST: SAME MODELS, BOTH GROUPS OF PATIENTS"); print("="*84)
m=pd.read_csv(os.path.join(D,"cv_mass_twostream_oof.csv")); m["_k"]=m["img"].map(stem)
cv=d.merge(m[["_k","prob"]].drop_duplicates("_k"),on="_k")
print("  Using the SAME cross-validation models, scored out-of-fold:")
for nm,s in (("official-TRAIN patients",cv[cv.sp.eq("train")]),
             ("official-TEST  patients",cv[cv.sp.eq("test")])):
    print("    %-26s n=%4d  malig %4.1f%%  AUC %.4f"
          % (nm,len(s),100*s.y.mean(),roc_auc_score(s.y,s.prob)))
a_tr=roc_auc_score(cv[cv.sp.eq("train")].y,cv[cv.sp.eq("train")].prob)
a_te=roc_auc_score(cv[cv.sp.eq("test")].y, cv[cv.sp.eq("test")].prob)
print("    difference (test minus train) %+.4f" % (a_te-a_tr))
print("\n  If the official-TEST patients score HIGHER under identical models, the official")
print("  test set is genuinely an easier draw and part of the official-vs-CV gap is case mix.")
print("  If they score the same or lower, the official test set is not anomalously easy.")

print("\n  per-fold CV AUC (is the official test AUC of 0.8769 unusual?)")
for k in sorted(cv.fold.dropna().unique()):
    s=cv[cv.fold==k]
    print("    fold %d  n=%4d  AUC %.4f" % (int(k),len(s),roc_auc_score(s.y,s.prob)))

PART 1 — CASE MIX: OFFICIAL TRAIN vs OFFICIAL TEST
  variable                   train          test           difference
  n regions                  1318           378           
  malignant %                48.3           38.9           -9.4 pts
  mean subtlety (1-5)        3.966          3.786          -0.180
  mean density (1-4)         2.203          2.397          +0.193
  mean mask Dice             0.895          0.906          +0.012

  BI-RADS distribution (%% of partition)
  BI-RADS    train      test       difference
  0          9.8        8.7        -1.1
  1          0.1        0.5        +0.5
  2          5.8        3.7        -2.1
  3          21.2       22.5       +1.3
  4          40.4       44.7       +4.3
  5          22.7       19.8       -2.8

PART 2 — ARE THE DIFFERENCES SIGNIFICANT?
  subtlety       Mann-Whitney p = 0.0084   DIFFERENT
  density        Mann-Whitney p = 0.0002   DIFFERENT
  mask Dice      Mann-Whitney p = 0.0003   DIFFERENT
  BI-RADS        chi-squ

In [2]:
import os, shutil
D = "/root/autodl-tmp/CBIS"
for f in ["cv_mass_twostream_officialsplit_oof.csv", "cv_mass_twostream_officialtrain_oof.csv",
          "cv_mass_twostream_calcpre_officialsplit_oof.csv",
          "cv_mass_twostream_calcpre_officialtrain_oof.csv"]:
    p = os.path.join(D, f)
    if os.path.exists(p):
        shutil.copy(p, p.replace(".csv", "_cvmasks.csv")); print("backed up", f)

backed up cv_mass_twostream_officialsplit_oof.csv
backed up cv_mass_twostream_calcpre_officialsplit_oof.csv
backed up cv_mass_twostream_calcpre_officialtrain_oof.csv


In [3]:
import os, shutil
D = "/root/autodl-tmp/CBIS"
for f in ["cv_mass_officialsplit_oof.csv",
          "cv_mass_twostream_officialsplit_oof.csv",
          "cv_mass_twostream_officialtrain_oof.csv"]:
    p = os.path.join(D, f)
    if os.path.exists(p):
        shutil.copy(p, p.replace(".csv", "_cvmasks.csv")); print("backed up", f)

backed up cv_mass_officialsplit_oof.csv
backed up cv_mass_twostream_officialsplit_oof.csv


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 20A-OFFICIAL — WIDE-CONTEXT MASS CROPS from the OFFICIAL-SPLIT MASKS
#   Identical geometry to your original cell 20A. The only changes:
#     PM  -> predmasks_mass_official      (masks from the official-split segmenter)
#     OUT -> crops_wide_mass_official     (your crops_wide_mass/ is untouched)
#   The projected mask is PREDICTED, never ground truth.
#   CPU only. ~20-30 min.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D    = "/root/autodl-tmp/CBIS"
JP   = os.path.join(D, "jpeg")
LES  = "mass"
PM   = os.path.join(D, "predmasks_%s_official" % LES)              # <-- OFFICIAL masks
OUT  = os.path.join(D, "crops_wide_%s_official" % LES); os.makedirs(OUT, exist_ok=True)
FIG  = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)

S          = 512
WIDE_MULT  = 1.75

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
for c in ["img", "msk", "full_series", "mask_series"]:
    assert c in d.columns, "column '%s' missing from unified_folds_%s.csv" % (c, LES)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))

# ═══════════════════ VERIFICATION ═══════════════════
print("="*66); print("VERIFICATION"); print("="*66)
assert os.path.isdir(PM), "official mask folder not found: %s\nRun the segmentation cell first." % PM
npm = int(d["predmask"].apply(os.path.exists).sum())
print("  mask source        : %s" % PM)
print("  official masks     : %d / %d present" % (npm, len(d)))
assert npm == len(d), "%d official masks missing — the segmentation cell did not finish" % (len(d)-npm)
_te = d["official_split"].astype(str).str.lower().str.contains("test")
print("  official split     : train %d | test %d regions" % ((~_te).sum(), _te.sum()))
print("  patients in BOTH   : %d" % len(set(d.patient_id[~_te]) & set(d.patient_id[_te])))
print("  output folder      : %s" % OUT)
print("  (this cell only re-crops; the masks it projects are PREDICTED, never ground truth)")
print("="*66 + "\n")


def series_files(uid):
    p = os.path.join(JP, str(uid))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []


def read_full(uid):
    best, ba = None, -1
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > ba:
            best, ba = im, im.size
    return best


def read_maskseries(uid, ref_shape):
    """mask series can contain BOTH a crop and the mask - pick by shape + binariness"""
    c = []
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None:
            continue
        score = 2.0 * float(im.shape == ref_shape) + float(((im < 20) | (im > 235)).mean())
        c.append((score, im))
    if not c:
        return None
    c.sort(key=lambda z: -z[0])
    return c[0][1]


def square_cut(im, cy, cx, side, interp, out=S):
    """square crop centred at (cy,cx) with zero-padding beyond the image edge"""
    s  = int(round(side))
    y0 = int(round(cy - s / 2.0)); x0 = int(round(cx - s / 2.0))
    y1, x1 = y0 + s, x0 + s
    ty0, tx0 = max(0, -y0), max(0, -x0)
    ty1, tx1 = max(0, y1 - im.shape[0]), max(0, x1 - im.shape[1])
    sub = im[max(y0, 0):min(y1, im.shape[0]), max(x0, 0):min(x1, im.shape[1])]
    if sub.size == 0:
        return None
    if ty0 or tx0 or ty1 or tx1:
        sub = cv2.copyMakeBorder(sub, ty0, ty1, tx0, tx1, cv2.BORDER_CONSTANT, value=0)
    return cv2.resize(sub, (out, out), interpolation=interp)


rows, skip = [], {}
diag = {"ratio": [], "cov_old": [], "cov_new": [], "side": []}
t0 = time.time()

for i, r in d.iterrows():
    if (i + 1) % 200 == 0:
        print("   %4d/%d   (%.0fs)" % (i + 1, len(d), time.time() - t0), flush=True)

    old_m = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    if old_m is None:
        skip["no existing mask"] = skip.get("no existing mask", 0) + 1; continue
    cov_old = float((old_m > 127).mean())
    if cov_old < 1e-4:
        skip["empty existing mask"] = skip.get("empty existing mask", 0) + 1; continue

    full = read_full(r["full_series"])
    if full is None:
        skip["no full jpg"] = skip.get("no full jpg", 0) + 1; continue
    fm = read_maskseries(r["mask_series"], full.shape)
    if fm is None:
        skip["no mask jpg"] = skip.get("no mask jpg", 0) + 1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)

    ys, xs = np.where(fm > 127)
    if len(ys) < 20:
        skip["mask too small"] = skip.get("mask too small", 0) + 1; continue
    A  = float(len(ys))
    cy, cx = 0.5 * (ys.min() + ys.max()), 0.5 * (xs.min() + xs.max())
    bmax = float(max(ys.max() - ys.min(), xs.max() - xs.min()) + 1)

    side_tight = float(np.sqrt(A / cov_old))
    side_tight = float(np.clip(side_tight, 1.1 * bmax, 5.0 * bmax))
    side_wide  = side_tight * WIDE_MULT

    img_w = square_cut(full, cy, cx, side_wide, cv2.INTER_AREA)
    if img_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    # ---- project the PREDICTED (official-split) mask into the wide frame ----
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if pm is None:
        skip["no predmask"] = skip.get("no predmask", 0) + 1; continue
    st  = max(int(round(side_tight)), 8)
    pmb = cv2.resize((pm > 127).astype(np.uint8) * 255, (st, st),
                     interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros(full.shape, np.uint8)
    ty0 = int(round(cy - st / 2.0)); tx0 = int(round(cx - st / 2.0))
    sy0, sx0 = max(ty0, 0), max(tx0, 0)
    sy1, sx1 = min(ty0 + st, full.shape[0]), min(tx0 + st, full.shape[1])
    if sy1 > sy0 and sx1 > sx0:
        canvas[sy0:sy1, sx0:sx1] = pmb[sy0 - ty0:sy1 - ty0, sx0 - tx0:sx1 - tx0]
    msk_w = square_cut(canvas, cy, cx, side_wide, cv2.INTER_NEAREST)
    if msk_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    stem = os.path.basename(str(r["img"])).replace("_img.png", "")
    pi = os.path.join(OUT, stem + "_img.png")
    pp = os.path.join(OUT, stem + "_pred.png")
    cv2.imwrite(pi, img_w)
    cv2.imwrite(pp, msk_w)

    gtw = square_cut(fm, cy, cx, side_wide, cv2.INTER_NEAREST)
    diag["ratio"].append(side_tight / bmax)
    diag["cov_old"].append(cov_old)
    diag["cov_new"].append(float((gtw > 127).mean()) if gtw is not None else np.nan)
    diag["side"].append(side_wide)
    rows.append((i, pi, pp))

print("\ngenerated %d / %d   (%.1f min)" % (len(rows), len(d), (time.time() - t0) / 60))
if skip:
    print("skipped:", skip)
assert len(rows) == len(d), \
    "only %d of %d wide crops were produced — CELL D/H assert on completeness" % (len(rows), len(d))

ok = pd.DataFrame(diag)
print("\n" + "=" * 66)
print("CROP GEOMETRY CHECK")
print("=" * 66)
print("  solved crop width / lesion width     median %.2f   (expect ~1.5-2.0)"
      % np.median(ok["ratio"]))
print("  lesion coverage, tight crops         median %.1f%%" % (100 * np.median(ok["cov_old"])))
print("  lesion coverage, WIDE crops          median %.1f%%   (expect ~%.1f%%)"
      % (100 * np.nanmedian(ok["cov_new"]), 100 * np.median(ok["cov_old"]) / WIDE_MULT ** 2))
print("  native pixels of the wide window     median %.0f" % np.median(ok["side"]))
print("  fraction of wide windows >= 512 px   %.1f%%"
      % (100 * float((np.array(ok["side"]) >= S).mean())))

sel = [r_[0] for r_ in rows[:: max(1, len(rows) // 4)]][:4]
fig, ax = plt.subplots(2, len(sel), figsize=(4 * len(sel), 8))
ax = np.atleast_2d(ax)
for j, gi in enumerate(sel):
    o  = cv2.imread(str(d.iloc[gi]["img"]), cv2.IMREAD_GRAYSCALE)
    om = cv2.imread(str(d.iloc[gi]["predmask"]), cv2.IMREAD_GRAYSCALE)
    stem = os.path.basename(str(d.iloc[gi]["img"])).replace("_img.png", "")
    w  = cv2.imread(os.path.join(OUT, stem + "_img.png"), cv2.IMREAD_GRAYSCALE)
    wm = cv2.imread(os.path.join(OUT, stem + "_pred.png"), cv2.IMREAD_GRAYSCALE)
    for row, (im_, m_, ttl) in enumerate([(o, om, "tight crop"), (w, wm, "wide context")]):
        ax[row, j].imshow(im_, cmap="gray")
        if m_ is not None and (m_ > 127).any():
            ax[row, j].contour(m_ > 127, levels=[0.5], colors="lime", linewidths=1.4)
        ax[row, j].set_title("%s\n%s" % (ttl, "MALIGNANT" if d.iloc[gi]["label"] == 1 else "benign"),
                             fontsize=9)
        ax[row, j].axis("off")
plt.tight_layout()
fp = os.path.join(FIG, "widectx_check_official.png")
plt.savefig(fp, dpi=130, bbox_inches="tight"); plt.close()
print("\n  figure: %s" % fp)

keep = pd.DataFrame(rows, columns=["ridx", "wide_img", "wide_pred"]).set_index("ridx")
nd = d.loc[keep.index].copy()
nd["img"]      = keep["wide_img"].values
nd["predmask"] = keep["wide_pred"].values
out_csv = os.path.join(D, "unified_folds_%s_wide_official.csv" % LES)
nd.to_csv(out_csv, index=False)
print("  saved  %s   (%d rows, folds/roles preserved)" % (os.path.basename(out_csv), len(nd)))
print("\n  -> next: CELL 6-OFFICIAL v3, then CELL D v2 with the _official paths.")

VERIFICATION
  mask source        : /root/autodl-tmp/CBIS/predmasks_mass_official
  official masks     : 1696 / 1696 present
  official split     : train 1318 | test 378 regions
  patients in BOTH   : 0
  output folder      : /root/autodl-tmp/CBIS/crops_wide_mass_official
  (this cell only re-crops; the masks it projects are PREDICTED, never ground truth)

    200/1696   (24s)
    400/1696   (49s)
    600/1696   (72s)
    800/1696   (97s)
   1000/1696   (121s)
   1200/1696   (145s)
   1400/1696   (170s)
   1600/1696   (193s)

generated 1696 / 1696   (3.4 min)

CROP GEOMETRY CHECK
  solved crop width / lesion width     median 1.57   (expect ~1.5-2.0)
  lesion coverage, tight crops         median 23.0%
  lesion coverage, WIDE crops          median 7.5%   (expect ~7.5%)
  native pixels of the wide window     median 882
  fraction of wide windows >= 512 px   94.5%

  figure: /root/autodl-tmp/CBIS/figures/mass_final/widectx_check_official.png
  saved  unified_folds_mass_wide_official.csv   

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6-OFFICIAL v3 — MASS CLASSIFIER (DenseNet-121, mask-weighted dual
# pooling) on the OFFICIAL SPLIT, using the OFFICIAL-SPLIT MASKS.
#   Identical hyperparameters to your CV cell 19.
#   Resumable per fold. Saves test-side AND train-side predictions.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

TAG  = "v2_offmask"
CKPT = os.path.join(D, "ckpt_official"); os.makedirs(CKPT, exist_ok=True)
PM   = os.path.join(D, "predmasks_%s_official" % LES)      # <-- OFFICIAL masks

S, BATCH   = 512, 12
SEEDS      = [11, 22]
EPOCHS     = 22
FREEZE     = 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W = 1e-4, 2.0, 0.3
MULT, PATIENCE, ATT = 4, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
miss = (~d["predmask"].apply(os.path.exists)).sum()
assert miss == 0, "%d official masks missing in %s — run the segmentation cell first" % (miss, PM)
assert "role_of0" in d.columns, "run CELL A first"
_te = d["official_split"].astype(str).str.lower().str.contains("test")
for k in range(5):
    r = d["role_of%d" % k]
    assert ((r == "test") == _te).all(), "role_of%d test != official test" % k
    assert not (r.isin(["train","val"]) & _te).any(), "training on official TEST!"
print("masks   : %s" % PM)
print("OFFICIAL SPLIT | train %d | test %d regions | %d test lesions"
      % ((~_te).sum(), _te.sum(), d.loc[_te, "lesion_key"].nunique()))

def ck_path(f): return os.path.join(CKPT, "%s_f%d.npz" % (TAG, f))
def ck_save(f, te, pt, va, pv):
    np.savez(ck_path(f),
             te_img=d.iloc[te]["img"].values.astype(str), te_prob=np.asarray(pt, np.float64),
             va_img=d.iloc[va]["img"].values.astype(str), va_prob=np.asarray(pv, np.float64))
def ck_load(f, te):
    p = ck_path(f)
    if not os.path.exists(p): return None
    try:
        z = np.load(p, allow_pickle=True)
        if list(z["te_img"]) != list(d.iloc[te]["img"].values.astype(str)):
            print("  (stale checkpoint fold %d ignored)" % f); return None
        return z
    except Exception as e:
        print("  (unreadable checkpoint fold %d: %s)" % (f, str(e)[:40])); return None
print("checkpoints on disk: %s" % [f for f in range(5) if os.path.exists(ck_path(f))])

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    if k in CACHE: continue
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((S, S), np.uint8) if im is None else im
    if im.shape != (S, S): im = cv2.resize(im, (S, S))
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    pm = (np.zeros((S, S), np.uint8) if pm is None
          else (cv2.resize(pm, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CACHE[k] = (_clahe.apply(im), pm)
print("cached %d images in %.0fs" % (len(CACHE), time.time() - t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx=np.asarray(idx); self.aug=aug; self.mult=mult if aug else 1; self.tta=tta
    def __len__(self): return len(self.idx)*self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        img, msk = CACHE[str(r["img"])]; img, msk = img.copy(), msk.copy()
        if self.aug:
            if np.random.rand() < 0.5: img, msk = img[:, ::-1], msk[:, ::-1]
            if np.random.rand() < 0.5: img, msk = img[::-1, :], msk[::-1, :]
            k = np.random.randint(4)
            if k: img, msk = np.rot90(img, k), np.rot90(msk, k)
            img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
            if np.random.rand() < 0.7:
                M = cv2.getRotationMatrix2D((S/2, S/2), np.random.uniform(-25,25),
                                            np.random.uniform(0.90,1.12))
                img = cv2.warpAffine(img, M, (S,S), flags=cv2.INTER_LINEAR,
                                     borderMode=cv2.BORDER_REFLECT)
                msk = cv2.warpAffine(msk, M, (S,S), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT)
            if np.random.rand() < 0.5:
                img = np.clip(img.astype(np.float32)*np.random.uniform(0.85,1.15)
                              + np.random.uniform(-12,12), 0, 255).astype(np.uint8)
        else:
            t = self.tta
            if   t == 1: img, msk = img[:, ::-1], msk[:, ::-1]
            elif t == 2: img, msk = img[::-1, :], msk[::-1, :]
            elif t == 3: img, msk = np.rot90(img, 2), np.rot90(msk, 2)
        img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
        im = img.astype(np.float32)/255.0
        x = ((np.stack([im, im, im], 0) - MEAN)/STD).astype(np.float32)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(x), torch.from_numpy(msk.astype(np.float32))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class GuidedNet(nn.Module):
    """DenseNet-121 + mask-weighted pooling AND unweighted whole-crop pooling."""
    def __init__(self, aux_meta, att=2.0):
        super().__init__()
        try: dn = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception:
            dn = models.densenet121(weights=None); print("  (ImageNet weights unavailable)")
        self.b, self.att = dn.features, att
        Fdim = 1024
        self.head = nn.Sequential(nn.Linear(Fdim*2, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(aux_meta.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fdim*2,128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
                                  for k in self.keys])
    def forward(self, x, mask):
        f = F.relu(self.b(x))
        m = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * m
        g_les = (f*w).sum((2,3)) / (w.sum((2,3)) + 1e-6)
        g_all = f.mean((2,3))
        g = torch.cat([g_les, g_all], 1)
        return self.head(g), [h(g) for h in self.aux]

def focal(logits, target, alpha):
    ce = F.cross_entropy(logits.float(), target, weight=alpha, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0,1,2,3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=20, shuffle=False, num_workers=0)
        ps = []
        for x, m, _, _ in ld:
            x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o, _ = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr]==0).sum()), float((y[tr]==1).sum())
    alpha = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = GuidedNet(meta, ATT).to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters(): p.requires_grad = False
    head_params = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(head_params, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, best_state, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        if ep == FREEZE + 1:
            for p in net.b.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.b.parameters(), "lr": LR_BACK},
                                     {"params": head_params,        "lr": LR_HEAD_FT}],
                                    weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.b.eval()
        for x, m, t, a in tl:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            m = m.to(DEV, non_blocking=True); t = t.to(DEV); a = a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(x, m)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            star = " *"
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star))
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in best_state.items()})
    pt = predict(net, te, tta=True)
    pv = predict(net, va, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, pv, best

y = d["label"].values
for k in FOLDS:
    role = d["role_of%d" % k]
    tr = np.where(role=="train")[0]; va = np.where(role=="val")[0]; te = np.where(role=="test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    if ck_load(k, te) is not None:
        print("\n### fold %d — already on disk, skipping" % k); continue
    print("\n### fold %d | train %d | val %d | test %d" % (k, len(tr), len(va), len(te)))
    pts, pvs, t0 = [], [], time.time()
    for sd in SEEDS:
        print("    seed %d" % sd)
        p, v, bv = train_one(tr, va, te, sd)
        pts.append(p); pvs.append(v)
        print("    seed %d done: best val %.4f | test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p)))
    ck_save(k, te, np.mean(pts, 0), va, np.mean(pvs, 0))
    print("  FOLD %d this fold AUC %.4f | CHECKPOINT SAVED (%.0fs)"
          % (k, roc_auc_score(y[te], np.mean(pts, 0)), time.time()-t0))

te0 = np.where(d["role_of0"].values == "test")[0]
parts, va_img, va_prob, have = [], [], [], []
for k in range(5):
    te = np.where(d["role_of%d" % k].values == "test")[0]
    z = ck_load(k, te)
    if z is None: continue
    parts.append(z["te_prob"]); have.append(k)
    va_img += list(z["va_img"]); va_prob += list(z["va_prob"])

print("\n" + "="*70)
print("MASS CLASSIFIER v2 (official masks)  |  folds completed: %s (%d/5)" % (have, len(have)))
print("="*70)
if not parts: raise SystemExit("no checkpoints yet")
oof = np.mean(parts, axis=0)
for i, f in enumerate(have):
    print("     after fold %d  ->  %.4f" % (f, roc_auc_score(y[te0], np.mean(parts[:i+1], axis=0))))
res = d.iloc[te0][["img","lesion_key","label"]].copy(); res["prob"] = oof
L = res.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
grid = np.linspace(0.05, 0.95, 181)
thr = grid[int(np.argmax([balanced_accuracy_score(L.y, (L.p > t).astype(int)) for t in grid]))]
pr = (L.p > thr).astype(int)
tn, fp, fn, tp = confusion_matrix(L.y, pr, labels=[0,1]).ravel()
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[te0], oof), len(te0)))
print("  per-lesion AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %d FN %d"
      % (roc_auc_score(L.y, L.p), 100*accuracy_score(L.y, pr),
         tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
print("  previous run with CV masks: 0.8652")

res.rename(columns={"label":"true"}).to_csv(
    os.path.join(D, "cv_mass_officialsplit_oof.csv"), index=False)
if va_img:
    tr_ = pd.DataFrame({"img": va_img, "prob": va_prob})
    tr_ = tr_.merge(d[["img","lesion_key","label"]], on="img", how="left").rename(columns={"label":"true"})
    tr_.to_csv(os.path.join(D, "cv_mass_v2_officialtrain_oof.csv"), index=False)
    print("  saved train-side file (%d rows)" % len(tr_))
print("  -> next: CELL D v2 with the official masks")
if len(have) < 5: print("  PARTIAL — re-run this cell later to add the missing folds.")

masks   : /root/autodl-tmp/CBIS/predmasks_mass_official
OFFICIAL SPLIT | train 1318 | test 378 regions | 223 test lesions
checkpoints on disk: []
cached 1696 images in 8s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1054 | val 264 | test 378
    seed 11
      ep  1  val-AUC 0.7407 *
      ep  2  val-AUC 0.5810
      ep  3  val-AUC 0.6325
      ep  4  val-AUC 0.7621 *
      ep  5  val-AUC 0.7877 *
      ep  6  val-AUC 0.8379 *
      ep  7  val-AUC 0.8567 *
      ep  8  val-AUC 0.8278
      ep  9  val-AUC 0.8603 *
      ep 10  val-AUC 0.8728 *
      ep 11  val-AUC 0.8534
      ep 12  val-AUC 0.8394
      ep 13  val-AUC 0.8689
      ep 14  val-AUC 0.8611
      ep 15  val-AUC 0.8606
      ep 16  val-AUC 0.8578
      ep 17  val-AUC 0.8592
      early stop
    seed 11 done: best val 0.8728 | test AUC 0.8398
    seed 22
      ep  1  val-AUC 0.7410 *
      ep  2  val-AUC 0.7246
      ep  3  val-AUC 0.7378
      ep  4  val-AUC 0.7750 *
      ep  5  val-A

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL D v2 — TWO-STREAM on the OFFICIAL SPLIT, OFFICIAL MASKS  [RESUMABLE]
#   Stream A: 512px tight crop + tight predicted mask
#   Stream B: 384px wide crop  + wide predicted mask
#   Architecture, seed and hyperparameters identical to your CV cell 47.
#   Saves test-side AND train-side predictions. Checkpoints every fold.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True

TAG  = "twostream_offmask"                                  # <- new tag: forces a fresh run
CKPT = os.path.join(D, "ckpt_official"); os.makedirs(CKPT, exist_ok=True)
WIDE = os.path.join(D, "crops_wide_%s_official" % LES)      # <- OFFICIAL wide crops
PM   = os.path.join(D, "predmasks_%s_official" % LES)       # <- OFFICIAL tight masks

ST, SW, BATCH = 512, 384, 8
SEEDS, EPOCHS, FREEZE = [11], 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png", ""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s + "_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_pred.png"))

print("=" * 70); print("VERIFICATION"); print("=" * 70)
print("  tight masks : %s" % PM)
print("  wide crops  : %s" % WIDE)
for col, name in [("tmask", "tight masks"), ("wimg", "wide images"), ("wmask", "wide masks")]:
    n = int(d[col].apply(os.path.exists).sum())
    print("  %-12s: %d / %d present" % (name, n, len(d)))
    assert n == len(d), "%d %s missing — run the segmentation cell and CELL 20A-OFFICIAL first" % (len(d)-n, name)
assert "role_of0" in d.columns, "run CELL A first"
_te = d["official_split"].astype(str).str.lower().str.contains("test")
for k in range(5):
    r = d["role_of%d" % k]
    assert ((r == "test") == _te).all(), "role_of%d test != official test" % k
    assert not (r.isin(["train","val"]) & _te).any(), "training on official TEST!"
print("  official split: train %d | test %d regions | %d test lesions"
      % ((~_te).sum(), _te.sum(), d.loc[_te, "lesion_key"].nunique()))
print("  patients in BOTH: %d" % len(set(d.patient_id[~_te]) & set(d.patient_id[_te])))
print("=" * 70)

def ck_path(f): return os.path.join(CKPT, "%s_f%d.npz" % (TAG, f))
def ck_save(f, te, pt, va, pv):
    np.savez(ck_path(f),
             te_img=d.iloc[te]["img"].values.astype(str), te_prob=np.asarray(pt, np.float64),
             va_img=d.iloc[va]["img"].values.astype(str), va_prob=np.asarray(pv, np.float64))
def ck_load(f, te):
    p = ck_path(f)
    if not os.path.exists(p): return None
    try:
        z = np.load(p, allow_pickle=True)
        if list(z["te_img"]) != list(d.iloc[te]["img"].values.astype(str)):
            print("  (stale checkpoint fold %d ignored)" % f); return None
        return z
    except Exception as e:
        print("  (unreadable checkpoint fold %d: %s)" % (f, str(e)[:40])); return None
print("checkpoints on disk: %s   (expect [] for a fresh run)"
      % [f for f in range(5) if os.path.exists(ck_path(f))])

cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size, size), np.uint8)
    if im.shape != (size, size):
        im = cv2.resize(im, (size, size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im > 127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d in %.0fs" % (len(CACHE), time.time() - t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug = np.asarray(idx), aug
        self.mult, self.tta = (mult if aug else 1), tta
    def __len__(self): return len(self.idx) * self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand() < .5, np.random.rand() < .5
            kk = np.random.randint(4)
            aff = np.random.rand() < .7
            ang, sc = np.random.uniform(-25, 25), np.random.uniform(.9, 1.12)
            itn = np.random.rand() < .5
            gg, bb = np.random.uniform(.85, 1.15), np.random.uniform(-12, 12)
            def T(im, mk, s):
                if fh: im, mk = im[:, ::-1], mk[:, ::-1]
                if fv: im, mk = im[::-1, :], mk[::-1, :]
                if kk: im, mk = np.rot90(im, kk), np.rot90(mk, kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2, s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s, s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s, s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32)*gg + bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta
            def V(im, mk):
                if   t == 1: return im[:, ::-1], mk[:, ::-1]
                elif t == 2: return im[::-1, :], mk[::-1, :]
                elif t == 3: return np.rot90(im, 2), np.rot90(mk, 2)
                return im, mk
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        out = []
        for im, mk in [(ti, tm), (wi, wm)]:
            im = np.ascontiguousarray(im).astype(np.float32) / 255.0
            out.append(torch.from_numpy(((np.stack([im]*3, 0) - MEAN) / STD).astype(np.float32)))
            out.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return out[0], out[1], out[2], out[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, att=2.0):
        super().__init__()
        def bb():
            try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        self.bt, self.bw, self.att = bb(), bb(), att
        Fd = 1024 * 4
        self.head = nn.Sequential(nn.Linear(Fd, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd, 128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[k]))
                                  for k in self.keys])
    def pool(self, b, x, m):
        f = F.relu(b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * mm
        return torch.cat([(f*w).sum((2,3)) / (w.sum((2,3)) + 1e-6), f.mean((2,3))], 1)
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([self.pool(self.bt, xt, mt), self.pool(self.bw, xw, mw)], 1)
        return self.head(g), [h(g) for h in self.aux]

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=12, shuffle=False, num_workers=0)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            xt, mt, xw, mw = xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o, _ = net(xt, mt, xw, mw)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    yy = d["label"].values
    n0, n1 = float((yy[tr] == 0).sum()), float((yy[tr] == 1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = TwoStream(meta, ATT).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for p_ in back: p_.requires_grad = False
    hp = [p_ for n_, p_ in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p_ in back: p_.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS - FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, mt, xw, mw, t_, a_ in tl:
            xt, mt = xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True)
            xw, mw = xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True)
            t_, a_ = t_.to(DEV), a_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(gg.float(), a_[:, h], ignore_index=-1)
                        for h, gg in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(yy[va], pv) if len(set(yy[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            star = " *"
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, tta=True)
    pv = predict(net, va, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, pv, best

y = d["label"].values
for k in FOLDS:
    role = d["role_of%d" % k]
    tr = np.where(role == "train")[0]; va = np.where(role == "val")[0]; te = np.where(role == "test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    if ck_load(k, te) is not None:
        print("\n### fold %d — already on disk, skipping" % k); continue
    print("\n### fold %d | train %d val %d test %d" % (k, len(tr), len(va), len(te)))
    pts, pvs, t0 = [], [], time.time()
    for sd in SEEDS:
        p_, v_, bv = train_one(tr, va, te, sd)
        pts.append(p_); pvs.append(v_)
        print("    seed %d: best val %.4f | test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p_)))
    ck_save(k, te, np.mean(pts, 0), va, np.mean(pvs, 0))
    print("  FOLD %d this fold AUC %.4f | CHECKPOINT SAVED (%.0fs)"
          % (k, roc_auc_score(y[te], np.mean(pts, 0)), time.time() - t0))

te0 = np.where(d["role_of0"].values == "test")[0]
parts, va_img, va_prob, have = [], [], [], []
for k in range(5):
    te = np.where(d["role_of%d" % k].values == "test")[0]
    z = ck_load(k, te)
    if z is None: continue
    parts.append(z["te_prob"]); have.append(k)
    va_img += list(z["va_img"]); va_prob += list(z["va_prob"])

print("\n" + "=" * 70)
print("TWO-STREAM (official masks)  |  folds completed: %s (%d/5)" % (have, len(have)))
print("=" * 70)
if not parts: raise SystemExit("no checkpoints yet")
oof = np.mean(parts, axis=0)
for i, f in enumerate(have):
    print("     after fold %d  ->  %.4f" % (f, roc_auc_score(y[te0], np.mean(parts[:i+1], axis=0))))
res = d.iloc[te0][["img", "lesion_key", "label"]].copy(); res["prob"] = oof
L = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean")).reset_index()
grid = np.linspace(0.05, 0.95, 181)
thr = grid[int(np.argmax([balanced_accuracy_score(L.y, (L.p > t).astype(int)) for t in grid]))]
pr = (L.p > thr).astype(int)
tn, fp, fn, tp = confusion_matrix(L.y, pr, labels=[0, 1]).ravel()
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[te0], oof), len(te0)))
print("  per-lesion AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %d FN %d"
      % (roc_auc_score(L.y, L.p), 100*accuracy_score(L.y, pr),
         tp/max(tp+fn, 1), tn/max(tn+fp, 1), fp, fn))
print("  previous run with CV masks: 0.9004")
print("  v2 with official masks:     0.8715")

res.rename(columns={"label": "true"}).to_csv(
    os.path.join(D, "cv_mass_twostream_officialsplit_oof.csv"), index=False)
if va_img:
    tr_ = pd.DataFrame({"img": va_img, "prob": va_prob})
    tr_ = tr_.merge(d[["img", "lesion_key", "label"]], on="img", how="left").rename(columns={"label": "true"})
    tr_.to_csv(os.path.join(D, "cv_mass_twostream_officialtrain_oof.csv"), index=False)
    print("  saved train-side file (%d rows)" % len(tr_))
print("  -> next: CELL H (calcpre), then CELL E v4")
if len(have) < 5: print("  PARTIAL — re-run this cell later to add the missing folds.")

VERIFICATION
  tight masks : /root/autodl-tmp/CBIS/predmasks_mass_official
  wide crops  : /root/autodl-tmp/CBIS/crops_wide_mass_official
  tight masks : 1696 / 1696 present
  wide images : 1696 / 1696 present
  wide masks  : 1696 / 1696 present
  official split: train 1318 | test 378 regions | 223 test lesions
  patients in BOTH: 0
checkpoints on disk: []   (expect [] for a fresh run)
cached 1696 in 17s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1054 val 264 test 378
      ep  1  val-AUC 0.7138 *
      ep  2  val-AUC 0.6447
      ep  3  val-AUC 0.6282
      ep  4  val-AUC 0.8085 *
      ep  5  val-AUC 0.8697 *
      ep  6  val-AUC 0.8760 *
      ep  7  val-AUC 0.8872 *
      ep  8  val-AUC 0.9039 *
      ep  9  val-AUC 0.8805
      ep 10  val-AUC 0.9057 *
      ep 11  val-AUC 0.8904
      ep 12  val-AUC 0.9001
      ep 13  val-AUC 0.8948
      ep 14  val-AUC 0.8889
      ep 15  val-AUC 0.9061 *
      ep 16  val-AUC 0.8973
      ep 17  val-AUC

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL E v4 — FINAL RESULT: fit on official TRAIN, report on official TEST
#   Pool restricted to models consistent with the OFFICIAL masks:
#     v2, two-stream, two-stream calcpre  (official masks)
#     imageonly                            (uses no mask at all)
#   handcrafted / endtoend are excluded: their features came from the
#   old CV masks, so they are not part of the official-mask pipeline.
#   Nothing is fitted on the 223 reported lesions.  CPU, ~30 s.
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

D          = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.70
LEAK_AUC   = 0.97
HEADLINE   = "cv_mass_twostream"          # the single model reported as the main result

MEMBERS = ["cv_mass_v2", "cv_mass_twostream",
           "cv_mass_twostream_calcpre", "cv_mass_imageonly"]
TEST_FILE = {"cv_mass_v2": "cv_mass_officialsplit_oof.csv"}
test_f  = lambda n: TEST_FILE.get(n, "%s_officialsplit_oof.csv" % n)
train_f = lambda n: "%s_officialtrain_oof.csv" % n

stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
d["_k"] = d["img"].map(stem)
assert d["_k"].is_unique
K2L = dict(zip(d["_k"], d["lesion_key"]))
K2Y = dict(zip(d["_k"], d["label"].astype(int)))
K2A = dict(zip(d["_k"], pd.to_numeric(d["assessment"], errors="coerce")))
K2S = dict(zip(d["_k"], d["official_split"].astype(str).str.lower()))

def load(fn, want_test):
    f = os.path.join(D, fn)
    if not os.path.exists(f): return None
    m = pd.read_csv(f); m["_k"] = m["img"].map(stem)
    m = m[m["_k"].isin(K2L)].copy()
    if not len(m): return None
    m["lesion_key"] = m["_k"].map(K2L); m["y"] = m["_k"].map(K2Y)
    m["a"] = m["_k"].map(K2A);          m["sp"] = m["_k"].map(K2S)
    m = m[m["sp"].str.contains("test") == want_test]
    if not len(m): return None
    g = m.groupby("lesion_key").agg(p=("prob","mean"), y=("y","max"),
                                    a=("a","first")).reset_index()
    g["a"] = pd.to_numeric(g.a, errors="coerce").fillna(4).astype(int).clip(0,5)
    return g

names, TE, TR = [], None, None
for nm in MEMBERS:
    t, r = load(test_f(nm), True), load(train_f(nm), False)
    if t is None: print("  absent (no test file) : %s" % nm); continue
    if r is None: print("  EXCLUDED (no train file): %s" % nm); continue
    assert roc_auc_score(t.y, t.p) < LEAK_AUC, "leakage guard on %s" % nm
    names.append(nm)
    TE = t[["lesion_key","y","a"]].copy() if TE is None else TE
    TR = r[["lesion_key","y","a"]].copy() if TR is None else TR
    TE = TE.merge(t[["lesion_key","p"]].rename(columns={"p":nm}), on="lesion_key")
    TR = TR.merge(r[["lesion_key","p"]].rename(columns={"p":nm}), on="lesion_key")
assert names, "no member has both a train-side and a test-side file"
assert HEADLINE in names, "%s is missing — cannot report the single-model headline" % HEADLINE

yte, ate = TE.y.values.astype(int), TE.a.values
ytr, atr = TR.y.values.astype(int), TR.a.values
assert not (set(TE.lesion_key) & set(TR.lesion_key)), "a lesion is in both sets"

print("\n%-30s %-11s %s" % ("member", "TRAIN AUC", "TEST AUC"))
print("-" * 56)
for nm in names:
    print("%-30s %-11.4f %.4f"
          % (nm, roc_auc_score(ytr, TR[nm].values), roc_auc_score(yte, TE[nm].values)))
print("\nTRAIN %d lesions (%.0f%% malignant) | TEST %d lesions (%.0f%% malignant)"
      % (len(ytr), 100*ytr.mean(), len(yte), 100*yte.mean()))

GRID = np.round(np.arange(0.02, 0.99, 0.01), 3)
def acc_of(y, yh): return (yh == y).mean()
def fit_tau(y, p, a, floor, passes=12):
    tau = {g: 0.50 for g in np.unique(a)}
    ap = lambda t: (p >= np.array([t.get(g, 0.50) for g in a])).astype(int)
    b = acc_of(y, ap(tau))
    for _ in range(passes):
        moved = False
        for g in tau:
            for c in GRID:
                t2 = dict(tau); t2[g] = c; yh = ap(t2)
                if yh[y == 1].mean() < floor: continue
                s = acc_of(y, yh)
                if s > b + 1e-9: tau, b, moved = t2, s, True
        if not moved: break
    return tau
def show(name, yh):
    tn, fp, fn, tp = confusion_matrix(yte, yh, labels=[0,1]).ravel()
    pr, rc = tp/max(tp+fp,1), tp/max(tp+fn,1)
    f1, ac = 2*pr*rc/max(pr+rc,1e-9), (tp+tn)/len(yte)
    print("  %-42s acc %.1f%%  sens %.3f  spec %.3f  F1 %.3f  FP %2d  missed %2d"
          % (name, 100*ac, rc, tn/max(tn+fp,1), f1, fp, fn))
    return ac, f1, fn

def report(title, ptr, pte):
    print("\n" + "=" * 86); print(title); print("=" * 86)
    auc = roc_auc_score(yte, pte)
    gt  = GRID[int(np.argmax([acc_of(ytr, (ptr >= t).astype(int)) for t in GRID]))]
    show("single global threshold (%.2f, from TRAIN)" % gt, (pte >= gt).astype(int))
    tau = fit_tau(ytr, ptr, atr, SENS_FLOOR)
    ac, f1, fn = show("per-BI-RADS  [NOVELTY 2]  (from TRAIN)",
                      (pte >= np.array([tau.get(g, 0.50) for g in ate])).astype(int))
    print("\n  AUC %.4f | accuracy %.1f%% | F1 %.3f | missed %d" % (auc, 100*ac, f1, fn))
    print("  thresholds (from TRAIN): %s" % {int(k): float(v) for k, v in sorted(tau.items())})
    return auc, ac, f1, fn

# ── A. the single model — this is the headline ────────────────────────
a1 = report("A.  SINGLE MODEL — two-stream (official masks)",
            TR[HEADLINE].values, TE[HEADLINE].values)

# ── B. greedy ensemble — the upper bound ──────────────────────────────
mixT = lambda s: np.column_stack([TR[m].values for m in s]).mean(1)
mixE = lambda s: np.column_stack([TE[m].values for m in s]).mean(1)
chosen, best = [], -np.inf
print("\ngreedy forward selection on official TRAIN:")
while True:
    pk, pa = None, best
    for nm in names:
        if nm in chosen: continue
        a = roc_auc_score(ytr, mixT(chosen + [nm]))
        if a > pa + 1e-6: pk, pa = nm, a
    if pk is None: break
    chosen.append(pk); best = pa
    print("   + %-30s train AUC -> %.4f" % (pk, best))
a2 = report("B.  ENSEMBLE — %s" % " + ".join(chosen), mixT(chosen), mixE(chosen))

print("\n" + "=" * 86); print("SUMMARY"); print("=" * 86)
print("  %-34s %-8s %-9s %-8s %s" % ("row", "AUC", "accuracy", "F1", "missed"))
print("  %-34s %-8.4f %-9.1f %-8.3f %d" % ("single model (two-stream)", a1[0], 100*a1[1], a1[2], a1[3]))
print("  %-34s %-8.4f %-9.1f %-8.3f %d" % ("ensemble (%d members)" % len(chosen), a2[0], 100*a2[1], a2[2], a2[3]))
print("  %-34s %-8.4f %-9.1f %-8s %s" % ("your five-fold CV", 0.9089, 87.0, "-", "-"))
print("\n  Nothing above was fitted on the 223 reported lesions.")


member                         TRAIN AUC   TEST AUC
--------------------------------------------------------
cv_mass_v2                     0.8860      0.8715
cv_mass_twostream              0.9081      0.9043
cv_mass_twostream_calcpre      0.8933      0.8867
cv_mass_imageonly              0.8045      0.8092

TRAIN 782 lesions (48% malignant) | TEST 223 lesions (39% malignant)

A.  SINGLE MODEL — two-stream (official masks)
  single global threshold (0.50, from TRAIN) acc 85.2%  sens 0.816  spec 0.875  F1 0.811  FP 17  missed 16
  per-BI-RADS  [NOVELTY 2]  (from TRAIN)     acc 86.5%  sens 0.816  spec 0.897  F1 0.826  FP 14  missed 16

  AUC 0.9043 | accuracy 86.5% | F1 0.826 | missed 16
  thresholds (from TRAIN): {0: 0.52, 1: 0.5, 2: 0.85, 3: 0.58, 4: 0.5, 5: 0.02}

greedy forward selection on official TRAIN:
   + cv_mass_twostream              train AUC -> 0.9081
   + cv_mass_twostream_calcpre      train AUC -> 0.9115
   + cv_mass_v2                     train AUC -> 0.9134

B.  ENSEMB

In [8]:
import os, shutil
D = "/root/autodl-tmp/CBIS"
for f in ["cv_mass_twostream_officialsplit_oof.csv",
          "cv_mass_twostream_officialtrain_oof.csv"]:
    p = os.path.join(D, f); b = p.replace(".csv", "_1seed.csv")
    assert os.path.exists(b), "backup missing: %s" % b
    shutil.copy(b, p); print("restored", os.path.basename(f))

restored cv_mass_twostream_officialsplit_oof.csv
restored cv_mass_twostream_officialtrain_oof.csv


In [9]:
# ══════════════════════════════════════════════════════════════════════
# CELL E v4 — FINAL RESULT: fit on official TRAIN, report on official TEST
#   Pool restricted to models consistent with the OFFICIAL masks:
#     v2, two-stream, two-stream calcpre  (official masks)
#     imageonly                            (uses no mask at all)
#   handcrafted / endtoend are excluded: their features came from the
#   old CV masks, so they are not part of the official-mask pipeline.
#   Nothing is fitted on the 223 reported lesions.  CPU, ~30 s.
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

D          = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.70
LEAK_AUC   = 0.97
HEADLINE   = "cv_mass_twostream"          # the single model reported as the main result

MEMBERS = ["cv_mass_v2", "cv_mass_twostream",
           "cv_mass_twostream_calcpre", "cv_mass_imageonly"]
TEST_FILE = {"cv_mass_v2": "cv_mass_officialsplit_oof.csv"}
test_f  = lambda n: TEST_FILE.get(n, "%s_officialsplit_oof.csv" % n)
train_f = lambda n: "%s_officialtrain_oof.csv" % n

stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
d["_k"] = d["img"].map(stem)
assert d["_k"].is_unique
K2L = dict(zip(d["_k"], d["lesion_key"]))
K2Y = dict(zip(d["_k"], d["label"].astype(int)))
K2A = dict(zip(d["_k"], pd.to_numeric(d["assessment"], errors="coerce")))
K2S = dict(zip(d["_k"], d["official_split"].astype(str).str.lower()))

def load(fn, want_test):
    f = os.path.join(D, fn)
    if not os.path.exists(f): return None
    m = pd.read_csv(f); m["_k"] = m["img"].map(stem)
    m = m[m["_k"].isin(K2L)].copy()
    if not len(m): return None
    m["lesion_key"] = m["_k"].map(K2L); m["y"] = m["_k"].map(K2Y)
    m["a"] = m["_k"].map(K2A);          m["sp"] = m["_k"].map(K2S)
    m = m[m["sp"].str.contains("test") == want_test]
    if not len(m): return None
    g = m.groupby("lesion_key").agg(p=("prob","mean"), y=("y","max"),
                                    a=("a","first")).reset_index()
    g["a"] = pd.to_numeric(g.a, errors="coerce").fillna(4).astype(int).clip(0,5)
    return g

names, TE, TR = [], None, None
for nm in MEMBERS:
    t, r = load(test_f(nm), True), load(train_f(nm), False)
    if t is None: print("  absent (no test file) : %s" % nm); continue
    if r is None: print("  EXCLUDED (no train file): %s" % nm); continue
    assert roc_auc_score(t.y, t.p) < LEAK_AUC, "leakage guard on %s" % nm
    names.append(nm)
    TE = t[["lesion_key","y","a"]].copy() if TE is None else TE
    TR = r[["lesion_key","y","a"]].copy() if TR is None else TR
    TE = TE.merge(t[["lesion_key","p"]].rename(columns={"p":nm}), on="lesion_key")
    TR = TR.merge(r[["lesion_key","p"]].rename(columns={"p":nm}), on="lesion_key")
assert names, "no member has both a train-side and a test-side file"
assert HEADLINE in names, "%s is missing — cannot report the single-model headline" % HEADLINE

yte, ate = TE.y.values.astype(int), TE.a.values
ytr, atr = TR.y.values.astype(int), TR.a.values
assert not (set(TE.lesion_key) & set(TR.lesion_key)), "a lesion is in both sets"

print("\n%-30s %-11s %s" % ("member", "TRAIN AUC", "TEST AUC"))
print("-" * 56)
for nm in names:
    print("%-30s %-11.4f %.4f"
          % (nm, roc_auc_score(ytr, TR[nm].values), roc_auc_score(yte, TE[nm].values)))
print("\nTRAIN %d lesions (%.0f%% malignant) | TEST %d lesions (%.0f%% malignant)"
      % (len(ytr), 100*ytr.mean(), len(yte), 100*yte.mean()))

GRID = np.round(np.arange(0.02, 0.99, 0.01), 3)
def acc_of(y, yh): return (yh == y).mean()
def fit_tau(y, p, a, floor, passes=12):
    tau = {g: 0.50 for g in np.unique(a)}
    ap = lambda t: (p >= np.array([t.get(g, 0.50) for g in a])).astype(int)
    b = acc_of(y, ap(tau))
    for _ in range(passes):
        moved = False
        for g in tau:
            for c in GRID:
                t2 = dict(tau); t2[g] = c; yh = ap(t2)
                if yh[y == 1].mean() < floor: continue
                s = acc_of(y, yh)
                if s > b + 1e-9: tau, b, moved = t2, s, True
        if not moved: break
    return tau
def show(name, yh):
    tn, fp, fn, tp = confusion_matrix(yte, yh, labels=[0,1]).ravel()
    pr, rc = tp/max(tp+fp,1), tp/max(tp+fn,1)
    f1, ac = 2*pr*rc/max(pr+rc,1e-9), (tp+tn)/len(yte)
    print("  %-42s acc %.1f%%  sens %.3f  spec %.3f  F1 %.3f  FP %2d  missed %2d"
          % (name, 100*ac, rc, tn/max(tn+fp,1), f1, fp, fn))
    return ac, f1, fn

def report(title, ptr, pte):
    print("\n" + "=" * 86); print(title); print("=" * 86)
    auc = roc_auc_score(yte, pte)
    gt  = GRID[int(np.argmax([acc_of(ytr, (ptr >= t).astype(int)) for t in GRID]))]
    show("single global threshold (%.2f, from TRAIN)" % gt, (pte >= gt).astype(int))
    tau = fit_tau(ytr, ptr, atr, SENS_FLOOR)
    ac, f1, fn = show("per-BI-RADS  [NOVELTY 2]  (from TRAIN)",
                      (pte >= np.array([tau.get(g, 0.50) for g in ate])).astype(int))
    print("\n  AUC %.4f | accuracy %.1f%% | F1 %.3f | missed %d" % (auc, 100*ac, f1, fn))
    print("  thresholds (from TRAIN): %s" % {int(k): float(v) for k, v in sorted(tau.items())})
    return auc, ac, f1, fn

# ── A. the single model — this is the headline ────────────────────────
a1 = report("A.  SINGLE MODEL — two-stream (official masks)",
            TR[HEADLINE].values, TE[HEADLINE].values)

# ── B. greedy ensemble — the upper bound ──────────────────────────────
mixT = lambda s: np.column_stack([TR[m].values for m in s]).mean(1)
mixE = lambda s: np.column_stack([TE[m].values for m in s]).mean(1)
chosen, best = [], -np.inf
print("\ngreedy forward selection on official TRAIN:")
while True:
    pk, pa = None, best
    for nm in names:
        if nm in chosen: continue
        a = roc_auc_score(ytr, mixT(chosen + [nm]))
        if a > pa + 1e-6: pk, pa = nm, a
    if pk is None: break
    chosen.append(pk); best = pa
    print("   + %-30s train AUC -> %.4f" % (pk, best))
a2 = report("B.  ENSEMBLE — %s" % " + ".join(chosen), mixT(chosen), mixE(chosen))

print("\n" + "=" * 86); print("SUMMARY"); print("=" * 86)
print("  %-34s %-8s %-9s %-8s %s" % ("row", "AUC", "accuracy", "F1", "missed"))
print("  %-34s %-8.4f %-9.1f %-8.3f %d" % ("single model (two-stream)", a1[0], 100*a1[1], a1[2], a1[3]))
print("  %-34s %-8.4f %-9.1f %-8.3f %d" % ("ensemble (%d members)" % len(chosen), a2[0], 100*a2[1], a2[2], a2[3]))
print("  %-34s %-8.4f %-9.1f %-8s %s" % ("your five-fold CV", 0.9089, 87.0, "-", "-"))
print("\n  Nothing above was fitted on the 223 reported lesions.")


member                         TRAIN AUC   TEST AUC
--------------------------------------------------------
cv_mass_v2                     0.8860      0.8715
cv_mass_twostream              0.9081      0.9043
cv_mass_twostream_calcpre      0.8933      0.8867
cv_mass_imageonly              0.8045      0.8092

TRAIN 782 lesions (48% malignant) | TEST 223 lesions (39% malignant)

A.  SINGLE MODEL — two-stream (official masks)
  single global threshold (0.50, from TRAIN) acc 85.2%  sens 0.816  spec 0.875  F1 0.811  FP 17  missed 16
  per-BI-RADS  [NOVELTY 2]  (from TRAIN)     acc 86.5%  sens 0.816  spec 0.897  F1 0.826  FP 14  missed 16

  AUC 0.9043 | accuracy 86.5% | F1 0.826 | missed 16
  thresholds (from TRAIN): {0: 0.52, 1: 0.5, 2: 0.85, 3: 0.58, 4: 0.5, 5: 0.02}

greedy forward selection on official TRAIN:
   + cv_mass_twostream              train AUC -> 0.9081
   + cv_mass_twostream_calcpre      train AUC -> 0.9115
   + cv_mass_v2                     train AUC -> 0.9134

B.  ENSEMB

In [11]:
# ══════════════════════════════════════════════════════════════════════
# CONFIDENCE INTERVALS + ABLATION SIGNIFICANCE  (CPU, ~1 min)
#   DeLong 95% CI on AUC, DeLong paired tests between models,
#   bootstrap 95% CIs on accuracy / F1 / sensitivity / specificity
#   at the thresholds fitted on the official TRAINING partition.
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, confusion_matrix

D, SENS_FLOOR, NBOOT, SEED = "/root/autodl-tmp/CBIS", 0.70, 2000, 42
MODELS = {"two-stream":         "cv_mass_twostream",
          "v2 (single)":        "cv_mass_v2",
          "image-only":         "cv_mass_imageonly",
          "two-stream calcpre": "cv_mass_twostream_calcpre"}
TEST_FILE = {"cv_mass_v2": "cv_mass_officialsplit_oof.csv"}
tf = lambda n: TEST_FILE.get(n, "%s_officialsplit_oof.csv" % n)
rf = lambda n: "%s_officialtrain_oof.csv" % n

stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
d["_k"] = d["img"].map(stem)
K = {"lesion_key": dict(zip(d["_k"], d["lesion_key"])),
     "y":  dict(zip(d["_k"], d["label"].astype(int))),
     "a":  dict(zip(d["_k"], pd.to_numeric(d["assessment"], errors="coerce"))),
     "sp": dict(zip(d["_k"], d["official_split"].astype(str).str.lower()))}

def load(fn, want_test):
    f = os.path.join(D, fn)
    if not os.path.exists(f): return None
    m = pd.read_csv(f); m["_k"] = m["img"].map(stem)
    m = m[m["_k"].isin(K["lesion_key"])].copy()
    for c in K: m[c] = m["_k"].map(K[c])
    m = m[m["sp"].str.contains("test") == want_test]
    if not len(m): return None
    g = m.groupby("lesion_key").agg(p=("prob","mean"), y=("y","max"),
                                    a=("a","first")).reset_index()
    g["a"] = pd.to_numeric(g.a, errors="coerce").fillna(4).astype(int).clip(0,5)
    return g

TE, TR, have = None, None, []
for label, nm in MODELS.items():
    t, r = load(tf(nm), True), load(rf(nm), False)
    if t is None:
        print("  missing test file : %s" % label); continue
    have.append(label)
    TE = t[["lesion_key","y","a"]].copy() if TE is None else TE
    TE = TE.merge(t[["lesion_key","p"]].rename(columns={"p":label}), on="lesion_key")
    if r is not None:
        TR = r[["lesion_key","y","a"]].copy() if TR is None else TR
        TR = TR.merge(r[["lesion_key","p"]].rename(columns={"p":label}), on="lesion_key")
Y, A = TE.y.values.astype(int), TE.a.values
print("official TEST: %d lesions, %d malignant (%.0f%%)\n" % (len(Y), Y.sum(), 100*Y.mean()))

# ---------------- DeLong ----------------
def midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2

def _structural(y, plist):
    order = np.argsort(-y)
    y2 = y[order]; m = int(y2.sum()); n = len(y2) - m
    P = np.vstack([p[order] for p in plist])
    tx = np.vstack([midrank(P[r, :m]) for r in range(P.shape[0])])
    ty = np.vstack([midrank(P[r, m:]) for r in range(P.shape[0])])
    tz = np.vstack([midrank(P[r, :])  for r in range(P.shape[0])])
    aucs = tz[:, :m].sum(1)/m/n - (m + 1.0)/2/n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    return aucs, v01, v10, m, n

def delong_ci(y, p, alpha=0.05):
    a, v01, v10, m, n = _structural(y, [p])
    var = v01[0].var(ddof=1)/m + v10[0].var(ddof=1)/n
    z = norm.ppf(1 - alpha/2); se = np.sqrt(var)
    return a[0], max(0.0, a[0] - z*se), min(1.0, a[0] + z*se)

def delong_test(y, p1, p2):
    a, v01, v10, m, n = _structural(y, [p1, p2])
    S = np.cov(v01)/m + np.cov(v10)/n
    diff = a[0] - a[1]
    var = S[0,0] + S[1,1] - 2*S[0,1]
    if var <= 0: return diff, float("nan")
    z = diff/np.sqrt(var)
    return diff, 2*(1 - norm.cdf(abs(z)))

print("%-22s %-8s DeLong 95%% CI" % ("model", "AUC"))
print("-"*52)
for label in have:
    a, lo, hi = delong_ci(Y, TE[label].values)
    print("%-22s %-8.4f [%.4f, %.4f]" % (label, a, lo, hi))

print("\nablation comparisons (DeLong paired test on the same %d lesions)" % len(Y))
print("-"*72)
for a_, b_, what in [("image-only", "v2 (single)", "mask-weighted pooling"),
                     ("v2 (single)", "two-stream", "wide-context branch"),
                     ("image-only", "two-stream", "full pipeline vs unguided")]:
    if a_ in have and b_ in have:
        diff, p = delong_test(Y, TE[b_].values, TE[a_].values)
        ptxt = "n/a" if p != p else ("%.2e" % p if p < 1e-3 else "%.4f" % p)
        print("  %-28s %+.4f   p = %s" % (what, diff, ptxt))

# ---------------- bootstrap CIs at the TRAIN-fitted thresholds ----------------
HEAD = "two-stream"
assert HEAD in have and TR is not None and HEAD in TR.columns, "need the two-stream train file"
GRID = np.round(np.arange(0.02, 0.99, 0.01), 3)
ytr, atr, ptr = TR.y.values.astype(int), TR.a.values, TR[HEAD].values

def fit(y, p, a, floor, passes=12):
    tau = {g: 0.50 for g in np.unique(a)}
    ap = lambda t: (p >= np.array([t.get(g, 0.50) for g in a])).astype(int)
    b = (ap(tau) == y).mean()
    for _ in range(passes):
        moved = False
        for g in tau:
            for c in GRID:
                t2 = dict(tau); t2[g] = c; yh = ap(t2)
                if yh[y == 1].mean() < floor: continue
                s = (yh == y).mean()
                if s > b + 1e-9: tau, b, moved = t2, s, True
        if not moved: break
    return tau

tau = fit(ytr, ptr, atr, SENS_FLOOR)
P  = TE[HEAD].values
YH = (P >= np.array([tau.get(g, 0.50) for g in A])).astype(int)

def metrics(y, yh):
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    pr, rc = tp/max(tp+fp,1), tp/max(tp+fn,1)
    return dict(accuracy=(tp+tn)/len(y), sensitivity=rc,
                specificity=tn/max(tn+fp,1), F1=2*pr*rc/max(pr+rc,1e-9),
                FP=fp, missed=fn)
obs = metrics(Y, YH)

rng = np.random.default_rng(SEED)
boot = {k: [] for k in ["accuracy","sensitivity","specificity","F1","FP","missed","AUC"]}
N = len(Y)
for _ in range(NBOOT):
    idx = rng.integers(0, N, N)
    if len(set(Y[idx])) < 2: continue
    mm = metrics(Y[idx], YH[idx])
    for k in ["accuracy","sensitivity","specificity","F1","FP","missed"]:
        boot[k].append(mm[k])
    boot["AUC"].append(roc_auc_score(Y[idx], P[idx]))

print("\n%s at thresholds fitted on the official TRAINING partition" % HEAD)
print("%-14s %-10s bootstrap 95%% CI (%d resamples)" % ("metric", "value", NBOOT))
print("-"*60)
print("%-14s %-10.4f [%.4f, %.4f]"
      % ("AUC", roc_auc_score(Y, P), *np.percentile(boot["AUC"], [2.5, 97.5])))
for k in ["accuracy","sensitivity","specificity","F1"]:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    print("%-14s %-10.4f [%.4f, %.4f]" % (k, obs[k], lo, hi))
for k in ["FP","missed"]:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    print("%-14s %-10d [%d, %d]" % (k, obs[k], int(round(lo)), int(round(hi))))
print("\n  thresholds used: %s" % {int(a_): float(b_) for a_, b_ in sorted(tau.items())})
print("  DeLong and bootstrap AUC intervals should agree closely.")

official TEST: 223 lesions, 87 malignant (39%)

model                  AUC      DeLong 95% CI
----------------------------------------------------
two-stream             0.9043   [0.8626, 0.9461]
v2 (single)            0.8715   [0.8231, 0.9198]
image-only             0.8092   [0.7505, 0.8680]
two-stream calcpre     0.8867   [0.8417, 0.9318]

ablation comparisons (DeLong paired test on the same 223 lesions)
------------------------------------------------------------------------
  mask-weighted pooling        +0.0622   p = 6.95e-04
  wide-context branch          +0.0329   p = 0.0082
  full pipeline vs unguided    +0.0951   p = 1.75e-05

two-stream at thresholds fitted on the official TRAINING partition
metric         value      bootstrap 95% CI (2000 resamples)
------------------------------------------------------------
AUC            0.9043     [0.8594, 0.9431]
accuracy       0.8655     [0.8161, 0.9103]
sensitivity    0.8161     [0.7294, 0.8904]
specificity    0.8971     [0.8433, 0.94

In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 17 — HOW MUCH BI-RADS ACCURACY DOES THE DECISION LAYER NEED?
#
#   Thresholds are fitted on TRAIN using the true BI-RADS categories
#   (available offline, from the dataset). At TEST time the category is
#   deliberately corrupted, simulating a model-predicted category.
#   This answers whether an autonomous version of the layer could work.
#
#   Two corruption models:
#     RANDOM   - a wrong prediction lands on any other category (pessimistic)
#     ADJACENT - a wrong prediction lands on a neighbouring category
#                (realistic: predictors confuse 3 with 4, not 1 with 5)
#
#   No GPU, no training. ~1 min.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15
ACCS=[1.00,0.95,0.90,0.85,0.80,0.70,0.60,0.50]; REPS=300
RNG=np.random.default_rng(7)

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
TR=d[d.sp.eq("train")].copy(); TR["p"]=load("cv_mass_twostream_officialtrain_oof.csv").loc[TR._k].values
TE=d[d.sp.eq("test")].copy();  TE["p"]=load("cv_mass_twostream_officialsplit_oof.csv").loc[TE._k].values

# ── threshold machinery ───────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    A=int(((yh==1)&(y==1)).sum()); B=int(((yh==0)&(y==0)).sum())
    if A/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bA=A-int((cur&(y[i]==1)).sum()); bB=B-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bA+tg+bB+ng)/N; se=(bA+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(A+B)/N+1e-12:
                tau[g]=float(GRID[j]); A,B=int(bA+tg[j]),int(bB+ng[j]); mv=True
        if not mv: break
    return tau,(A+B)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

def roll(t,key):
    if key is None: return t.reset_index(drop=True)
    return t.groupby(key,sort=True).agg(p=("p","mean"),y=("y","max"),a=("a","max")).reset_index()

# ── corruption models ─────────────────────────────────────────────────────
def corrupt(a, acc, cats, mode, rng):
    """with probability (1-acc) replace the category"""
    out=a.copy()
    wrong=rng.random(len(a)) > acc
    idx=np.where(wrong)[0]
    for i in idx:
        if mode=="random":
            opts=[c for c in cats if c!=a[i]]
        else:                                   # adjacent
            opts=[c for c in (a[i]-1,a[i]+1) if c in cats]
            if not opts: opts=[c for c in cats if c!=a[i]]
        if opts: out[i]=rng.choice(opts)
    return out

# ── run ───────────────────────────────────────────────────────────────────
for LVL,KEY in (("ROI",None),("LESION","lesion_key")):
    tr,te=roll(TR,KEY),roll(TE,KEY)
    ytr,ptr,atr=tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int)
    yte,pte,ate=te.y.values.astype(int),te.p.values,te.a.values.astype(int)
    cats=sorted(set(atr)|set(ate))

    g1=fit_global(ytr,ptr,FLOOR)                       # no layer at all
    tau,g0=fit_bir(ytr,ptr,atr,FLOOR)                  # thresholds from TRUE train categories
    acc_global=((pte>=g1).astype(int)==yte).mean()
    acc_perfect=(ap(pte,ate,tau,g0)==yte).mean()

    print("\n"+"#"*86)
    print("# %s LEVEL — n=%d | thresholds %s"
          % (LVL,len(te),{int(k):round(v,3) for k,v in sorted(tau.items())}))
    print("#"*86)
    print("  no decision layer (one global threshold) : %.4f" % acc_global)
    print("  decision layer, TRUE categories          : %.4f   (+%.2f points)"
          % (acc_perfect,100*(acc_perfect-acc_global)))
    print("\n  now corrupting the TEST categories, as a predicted category would be:")
    print("  %-10s %-26s %s" % ("category","RANDOM errors","ADJACENT errors"))
    print("  %-10s %-26s %s" % ("accuracy","accuracy [5th-95th]  gain","accuracy [5th-95th]  gain"))
    breakeven={}
    for acc in ACCS:
        row=["%-10.2f" % acc]
        for mode in ("random","adjacent"):
            rng=np.random.default_rng(11)
            v=[]
            for _ in range(REPS):
                an=corrupt(ate,acc,cats,mode,rng)
                v.append((ap(pte,an,tau,g0)==yte).mean())
            v=np.array(v); m=v.mean()
            row.append("%.4f [%.3f-%.3f] %+5.2f " %
                       (m,np.percentile(v,5),np.percentile(v,95),100*(m-acc_global)))
            if mode not in breakeven and m<=acc_global: breakeven[mode]=acc
        print("  %s %s %s" % (row[0],row[1],row[2]))
    print("\n  break-even (category accuracy below which the layer stops helping):")
    for mode in ("random","adjacent"):
        b=breakeven.get(mode)
        print("    %-10s %s" % (mode, ("~%.2f" % b) if b else "never in the tested range — layer still helps at 0.50"))


######################################################################################
# ROI LEVEL — n=378 | thresholds {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}
######################################################################################
  no decision layer (one global threshold) : 0.7566
  decision layer, TRUE categories          : 0.8095   (+5.29 points)

  now corrupting the TEST categories, as a predicted category would be:
  category   RANDOM errors              ADJACENT errors
  accuracy   accuracy [5th-95th]  gain  accuracy [5th-95th]  gain
  1.00       0.8095 [0.810-0.810] +5.29  0.8095 [0.810-0.810] +5.29 
  0.95       0.8043 [0.796-0.812] +4.77  0.8039 [0.796-0.812] +4.73 
  0.90       0.7985 [0.788-0.807] +4.19  0.7983 [0.788-0.810] +4.17 
  0.85       0.7925 [0.778-0.804] +3.59  0.7932 [0.780-0.807] +3.66 
  0.80       0.7870 [0.772-0.802] +3.04  0.7871 [0.772-0.802] +3.05 
  0.70       0.7755 [0.759-0.794] +1.89  0.7773 [0.759-0.794] +2.07 
  0.

In [2]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 18 — ARE THE CROPS LESION-CENTRED?
#   If the mask fills a large, central fraction of the crop, the crop was
#   generated from the lesion annotation and the lesion is already located.
# ══════════════════════════════════════════════════════════════════════════
import os, pandas as pd, numpy as np
D=r"/root/autodl-tmp/CBIS"

f=os.path.join(D,"mask_geometry_audit.csv")
if os.path.exists(f):
    g=pd.read_csv(f)
    print("="*76); print("MASK GEOMETRY INSIDE THE CROP"); print("="*76)
    print("  rows: %d   columns: %s\n" % (len(g),list(g.columns)))
    for c in ("frac","bbox_frac","cx_off"):
        if c in g.columns:
            v=pd.to_numeric(g[c],errors="coerce").dropna()
            print("  %-12s mean %.4f | median %.4f | 5%% %.4f | 95%% %.4f"
                  % (c,v.mean(),v.median(),v.quantile(.05),v.quantile(.95)))
    print("\n  frac      = fraction of the crop occupied by the mask")
    print("  bbox_frac = fraction occupied by the mask's bounding box")
    print("  cx_off    = how far the mask centroid sits from the crop centre")
else:
    print("mask_geometry_audit.csv not found")

for f2 in ("crop_coverage_check.csv","crop_edge_check.csv"):
    p=os.path.join(D,f2)
    if os.path.exists(p):
        t=pd.read_csv(p)
        print("\n"+"="*76); print(f2); print("="*76)
        print("  columns: %s" % list(t.columns))
        print(t.describe().T.to_string())

print("\n"+"="*76); print("HOW TO READ IT"); print("="*76)
print("  mask fills a LARGE fraction (say >15-20%) and sits NEAR THE CENTRE")
print("     -> the crop was made from the lesion annotation.")
print("        Your network delineates a boundary; it does not detect the lesion.")
print("  mask is SMALL and OFF-CENTRE")
print("     -> the crop is loosely defined and the network is doing real localisation.")

MASK GEOMETRY INSIDE THE CROP
  rows: 3562   columns: ['img', 'abn', 'label', 'frac', 'bbox_frac', 'cx_off', 'cy_off', 'bright_in', 'bright_out', 'contrast']

  frac         mean 0.4030 | median 0.4053 | 5% 0.3091 | 95% 0.4920
  bbox_frac    mean 0.5991 | median 0.5922 | 5% 0.5892 | 95% 0.6553
  cx_off       mean 0.0021 | median 0.0012 | 5% -0.0419 | 95% 0.0493

  frac      = fraction of the crop occupied by the mask
  bbox_frac = fraction occupied by the mask's bounding box
  cx_off    = how far the mask centroid sits from the crop centre

crop_coverage_check.csv
  columns: ['lesion_key', 'pathology', 'area_full', 'area_crop', 'retained']
            count          mean           std           min           25%           50%          75%           max
area_full  1696.0  78978.133255  79427.752567   4020.000000  36535.500000  57087.500000  90774.75000  1.256482e+06
area_crop  1696.0  60132.683962  10311.112076  17007.000000  53943.750000  60379.500000  65632.75000  1.558230e+05
retaine